# Meghalaya Report

## Basic setup

Importing libraries, loading data, basic transformations, etc.

### Load libraries

In [1]:
# Load libraries
import os
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

# Configure Plotly renderer
# this enable chart in the exported HTML file
# https://github.com/microsoft/vscode-jupyter/issues/6999
pio.renderers.default = "notebook_connected"

In [2]:
# R Packages, libraries and utilities
import rpy2
import rpy2.robjects as ro
from rpy2.robjects import r, Formula, pandas2ri
from rpy2.robjects.packages import importr

In [3]:
# Check R version
r_version = ro.r('version')

In [4]:
pandas2ri.activate()
ro.r('library(survey)')
ro.r('library(base)')
ro.r('library(dplyr)')

R[write to console]: Loading required package: grid

R[write to console]: Loading required package: Matrix

R[write to console]: Loading required package: survival

R[write to console]: 
Attaching package: ‘survey’


R[write to console]: The following object is masked from ‘package:graphics’:

    dotchart


R[write to console]: 
Attaching package: ‘dplyr’


R[write to console]: The following objects are masked from ‘package:stats’:

    filter, lag


R[write to console]: The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




### Load data

In [5]:
# Load data
df = pd.read_csv('/Users/ggojedap/Documents/work/Sonder/07/meghalaya/data/meghalaya_segmentaton_2024_4.csv')

In [6]:
segment_colors = {
    "R2": "#72a622",
    "R3.1": "#adcd57",
    "R3.2": "#f5ff91",
    "R3.3": "#fdc551",
    "R4": "#ff9300",
    "U1": "#005392",
    "U3": "#5da5cd", 
    "U4": "#abe9ff"
}

colors_yes_no = {
    "Yes": "#04859b",
    "No": "#c7c1c2"
}

colors_yes_no_inverted = {
    "No": "#04859b",
    "Yes": "#c7c1c2"
}

### Basic transformations

In [7]:
# Data transformations 
# Remove the suffix '-MG.IND' from the 'segment' column
df['segment'] = df['segment'].str.replace('-MG.IND', '', regex=False)

In [8]:
segment_location = df.groupby(['segment','v025']).size().reset_index(name='count')
segment_location = segment_location.drop(columns=['count'])

In [9]:
# Global variables required in the R runtime:
# ------------------------------------------

# Assign the R DataFrame to an R variable
ro.globalenv['survey_data'] = df

# Define the survey design in R
ro.r('survey_design <- svydesign(ids=~v021, strata=~v023, weights=~weight, data=survey_data)')

/Users/ggojedap/.local/share/virtualenvs/meghalaya-VbfvQkMt/lib/python3.11/site-packages/rpy2/robjects/pandas2ri.py:65: UserWarning:

Error while trying to convert the column "w.working". Fall back to string conversion. The error is: Series can only be of one type, or None (and here we have <class 'float'> and <class 'str'>). If happening with a pandas DataFrame the method infer_objects() will normalize data types before conversion.

/Users/ggojedap/.local/share/virtualenvs/meghalaya-VbfvQkMt/lib/python3.11/site-packages/rpy2/robjects/pandas2ri.py:65: UserWarning:

Error while trying to convert the column "w.work.freq". Fall back to string conversion. The error is: Series can only be of one type, or None (and here we have <class 'float'> and <class 'str'>). If happening with a pandas DataFrame the method infer_objects() will normalize data types before conversion.

/Users/ggojedap/.local/share/virtualenvs/meghalaya-VbfvQkMt/lib/python3.11/site-packages/rpy2/robjects/pandas2ri.py:65: Us

### Main functions

In [10]:
# Function to create contingency tables by demand
def create_table(var_of_iterest, inverted = False):
    ro.r('current_data <- survey_data %>% filter(' + var_of_iterest + ' != "NA")')
    ro.r('current_design <- svydesign(ids=~v021, strata=~v023, weights=~weight, data=current_data)')
    ro.r('current_table <- as.data.frame(svytable(~segment + ' + var_of_iterest + ', design = current_design))')
    ro.r('current_table$segment <- as.character(current_table$segment)')
    inner_table = ro.r('current_table')
    with (ro.default_converter + pandas2ri.converter).context():
        inner_table = ro.conversion.get_conversion().rpy2py(inner_table)  
    
    inner_total = inner_table.groupby('segment')['Freq'].transform('sum')
    inner_table['percentage'] = inner_table['Freq'] / inner_total * 100
    if inverted == True:
        inner_table[var_of_iterest] = inner_table[var_of_iterest].astype(int).replace({0: 'No', 1: 'Yes'})
    else:
        inner_table[var_of_iterest] = inner_table[var_of_iterest].astype(int).replace({0: 'Yes', 1: 'No'})
        
    inner_table = inner_table.drop("Freq", axis=1)
    
    return inner_table

In [11]:
# Function to create contingency tables by demand
def create_table_categorical(var_of_iterest):
    ro.r('current_data <- survey_data %>% filter(' + var_of_iterest + ' != "NA")')
    ro.r('current_design <- svydesign(ids=~v021, strata=~v023, weights=~weight, data=current_data)')
    ro.r('current_table <- as.data.frame(svytable(~segment + ' + var_of_iterest + ', design = current_design))')
    ro.r('current_table$segment <- as.character(current_table$segment)')
    inner_table = ro.r('current_table')
    with (ro.default_converter + pandas2ri.converter).context():
        inner_table = ro.conversion.get_conversion().rpy2py(inner_table)  
    
    inner_total = inner_table.groupby('segment')['Freq'].transform('sum')
    inner_table['percentage'] = inner_table['Freq'] / inner_total * 100
    
    inner_table = inner_table.drop("Freq", axis=1)
    
    return inner_table

In [12]:
def create_figure_stacked_bars(data, var_of_interest, title, label_description, inverted = False):
    # Create a stacked bar chart using Plotly Express with percentages
    fig = px.bar(data,
                x='segment', 
                y='percentage', 
                color=var_of_interest, 
                text='percentage',
                color_discrete_map = colors_yes_no_inverted if inverted == True else colors_yes_no, 
                title=title, 
                labels={var_of_interest: label_description},
                barmode='stack')

    # Customize the layout to improve the appearance of the text labels
    fig.update_traces(texttemplate='%{text:.1f}%', textposition='inside')
    
    return fig

## 1. Overview

### 1.1. Segment distribution

In [13]:
# Create the "weighted" contingency table and convert it to a data frame
ro.r('current_table <- as.data.frame(svytable(~segment, design = survey_design))')
ro.r('current_table$segment <- as.character(current_table$segment)')

# retriving the table from R to Python
tbl_segment_size = ro.r('current_table')

# convert rpy2.robjects.vectors.DataFrame to pandas.DataFrame
with (ro.default_converter + pandas2ri.converter).context():
  tbl_segment_size = ro.conversion.get_conversion().rpy2py(tbl_segment_size)

In [14]:
# Calculate total count of all segments
segment_total_count = tbl_segment_size['Freq'].sum()

# Calculate percentage within each segment
tbl_segment_size['percentage'] = (tbl_segment_size['Freq'] / segment_total_count) * 100

# Merge counts and totals to calculate percentages
tbl_segment_size = pd.merge(tbl_segment_size, segment_location, on='segment').sort_values(["v025", "percentage"], ascending=[True, False])

# Remove count of weighted samples
tbl_segment_size = tbl_segment_size.drop("Freq", axis=1)

# Display the resulting table
tbl_segment_size

,segment,percentage,v025
3,R4,31.774180,rural
1,R3.1,20.465744,rural
2,R3.2,12.964190,rural
0,R2,11.864513,rural
5,U2,11.871912,urban
4,U1,8.181852,urban
6,U4,2.877608,urban


In [15]:
# Create a pie chart using Plotly Express
fig = px.pie(tbl_segment_size, values='percentage', names='segment', title='Segment Distribution', color=tbl_segment_size.segment, color_discrete_map=segment_colors)

# Show the plot
fig.show()

## 2. Differentiating Variables: Key Vunerability Measures Used in Clustering

### 2.1. Woman and her past experience


#### 2.1.1 Ideal number of daughters.


In [16]:
create_table_categorical("ideal.num.girl")

,segment,ideal.num.girl,percentage
1,R2,1-3,70.143041
2,R3.1,1-3,71.184156
3,R3.2,1-3,72.412129
4,R4,1-3,42.992070
5,U1,1-3,86.620517
6,U2,1-3,60.051630
7,U4,1-3,29.208384
8,R2,3+,10.931566
9,R3.1,3+,15.533057
10,R3.2,3+,6.827494


In [17]:
create_figure_stacked_bars(create_table_categorical("ideal.num.girl"), "ideal.num.girl", "Ideal number of daughters the respondent desires", "ideal number of daughters?").show()

#### 2.1.2 Religion


In [18]:
create_table("hindu", True)

,segment,hindu,percentage
1,R2,No,86.901390
2,R3.1,No,94.510716
3,R3.2,No,89.802172
4,R4,No,99.871850
5,U1,No,100.000000
6,U2,No,90.010678
7,U4,No,94.934570
8,R2,Yes,13.098610
9,R3.1,Yes,5.489284
10,R3.2,Yes,10.197828


In [19]:
create_figure_stacked_bars(create_table("hindu", True), "hindu", "Religion", "respondent's religion is Hindu?")

#### 2.1.3 Starting cohabitation and birth.


In [20]:
create_table_categorical("cohab.birth.months")

,segment,cohab.birth.months,percentage
1,R2,1: within a year,33.384553
2,R3.1,1: within a year,38.211433
3,R3.2,1: within a year,38.138116
4,R4,1: within a year,39.673506
5,U1,1: within a year,30.475473
6,U2,1: within a year,42.656004
7,U4,1: within a year,30.756168
8,R2,2: 1-2 years,57.731896
9,R3.1,2: 1-2 years,48.760671
10,R3.2,2: 1-2 years,50.079144


In [21]:
create_figure_stacked_bars(create_table_categorical("cohab.birth.months"), "cohab.birth.months", "Time between starting cohabitation and first birth", "time between starting cohabitation and first birth?").show()

#### 2.1.4 Highest level of education attained.


In [22]:
create_table_categorical("ed.level")

,segment,ed.level,percentage
1,R2,0: no education,9.634350
2,R3.1,0: no education,2.604383
3,R3.2,0: no education,13.380996
4,R4,0: no education,25.497651
5,U1,0: no education,0.000000
6,U2,0: no education,1.525219
7,U4,0: no education,13.396078
8,R2,1: incomplete primary,21.167271
9,R3.1,1: incomplete primary,12.963427
10,R3.2,1: incomplete primary,18.143532


In [23]:
create_figure_stacked_bars(create_table_categorical("ed.level"), "ed.level", "Highest level of education attained", "level")

#### 2.1.5 Respondent's caste.

In [24]:
create_table_categorical("caste")

,segment,caste,percentage
1,R2,1: Schedule caste/tribe,91.677897
2,R3.1,1: Schedule caste/tribe,97.099906
3,R3.2,1: Schedule caste/tribe,100.000000
4,R4,1: Schedule caste/tribe,98.699228
5,U1,1: Schedule caste/tribe,100.000000
6,U2,1: Schedule caste/tribe,94.878409
7,U4,1: Schedule caste/tribe,100.000000
8,R2,2: obc,4.173855
9,R3.1,2: obc,0.449112
10,R3.2,2: obc,0.000000


In [25]:
create_figure_stacked_bars(create_table_categorical("caste"), "caste", "Caste", "respondent's caste?")

#### 2.1.6 The respondent has any education (primary level or above).

In [26]:
create_table("ed.any", True)

,segment,ed.any,percentage
1,R2,No,9.634350
2,R3.1,No,2.604383
3,R3.2,No,13.380996
4,R4,No,25.497651
5,U1,No,0.000000
6,U2,No,1.525219
7,U4,No,13.396078
8,R2,Yes,90.365650
9,R3.1,Yes,97.395617
10,R3.2,Yes,86.619004


In [27]:
create_figure_stacked_bars(create_table("ed.any", True), "ed.any", "The respondent has any education (primary level or above)", "Has any education?")

#### 2.1.7 Age group at which the respondent first had sexual intercourse.

In [28]:
create_table_categorical("first_sex_age_group")

,segment,first_sex_age_group,percentage
1,R2,<18,15.445570
2,R3.1,<18,21.999428
3,R3.2,<18,18.623111
4,R4,<18,34.331608
5,U1,<18,10.988687
6,U2,<18,14.448827
7,U4,<18,46.883458
8,R2,>25,25.527528
9,R3.1,>25,19.084211
10,R3.2,>25,22.767722


In [29]:
create_figure_stacked_bars(create_table_categorical("first_sex_age_group"), "first_sex_age_group", "Age group at which the respondent first had sexual intercourse", "age group")

#### 2.1.8 The ideal number of sons the respondent desires.

In [30]:
create_table_categorical("ideal.num.boy")

,segment,ideal.num.boy,percentage
1,R2,1-3,75.224405
2,R3.1,1-3,76.147714
3,R3.2,1-3,76.874570
4,R4,1-3,39.835230
5,U1,1-3,96.733335
6,U2,1-3,58.016220
7,U4,1-3,24.221299
8,R2,3+,6.302222
9,R3.1,3+,9.957971
10,R3.2,3+,2.661635


In [31]:
create_figure_stacked_bars(create_table_categorical("ideal.num.boy"), "ideal.num.boy", "The ideal number of sons the respondent desires.", "ideal number of sons?")

#### 2.1.9 The respondent prefers more sons than daughters.

In [32]:
create_table("male.child.pref", True)

,segment,male.child.pref,percentage
1,R2,No,87.696783
2,R3.1,No,84.324827
3,R3.2,No,84.404822
4,R4,No,79.276380
5,U1,No,93.710380
6,U2,No,84.625195
7,U4,No,66.645900
8,R2,Yes,12.303217
9,R3.1,Yes,15.675173
10,R3.2,Yes,15.595178


In [33]:
create_figure_stacked_bars(create_table("male.child.pref", True), "male.child.pref", "The respondent prefers more sons than daughters", "prefers more sons than daughters?")

#### 2.1.10. Age group at which the respondent first married or started cohabiting.

In [34]:
create_table_categorical("marrage_age_group")

,segment,marrage_age_group,percentage
1,R2,<18,15.859134
2,R3.1,<18,23.817831
3,R3.2,<18,23.335719
4,R4,<18,36.659953
5,U1,<18,11.688574
6,U2,<18,16.157967
7,U4,<18,56.640664
8,R2,>25,17.984982
9,R3.1,>25,15.283142
10,R3.2,>25,14.543792


In [35]:
create_figure_stacked_bars(create_table_categorical("marrage_age_group"), "marrage_age_group", "Age group at which the respondent first married or started cohabiting", "age group for first married or started cohabiting")

#### 2.1.11 The respondent has more sons than daughters.

In [36]:
create_table("more.sons", True)

,segment,more.sons,percentage
1,R2,No,62.646361
2,R3.1,No,65.695749
3,R3.2,No,66.408194
4,R4,No,57.075093
5,U1,No,76.934073
6,U2,No,71.598514
7,U4,No,57.414685
8,R2,Yes,37.353639
9,R3.1,Yes,34.304251
10,R3.2,Yes,33.591806


In [37]:
create_figure_stacked_bars(create_table("more.sons", True), "more.sons", "The respondent has more sons than daughters.", "more sons than daughters?")

### 2.2. Household relationships


#### 2.2.1. Kids in the household.


In [38]:
create_table_categorical("num.kids.house.cat")

,segment,num.kids.house.cat,percentage
1,R2,0,0.000000
2,R3.1,0,0.123996
3,R3.2,0,1.862826
4,R4,0,0.077612
5,U1,0,0.000000
6,U2,0,0.000000
7,U4,0,0.000000
8,R2,1,33.381014
9,R3.1,1,34.106748
10,R3.2,1,39.759357


In [39]:
create_figure_stacked_bars(create_table_categorical("num.kids.house.cat"), "num.kids.house.cat", "Number of kids in the household", "Number of kids in the household?")

#### 2.2.2. Number of children in the household.


In [40]:
create_table_categorical("cohab.child.cat")

,segment,cohab.child.cat,percentage
1,R2,0,0.000000
2,R3.1,0,0.123996
3,R3.2,0,1.862826
4,R4,0,0.195159
5,U1,0,8.563758
6,U2,0,14.776580
7,U4,0,6.614292
8,R2,1,35.642193
9,R3.1,1,27.205881
10,R3.2,1,39.579206


In [41]:
create_figure_stacked_bars(create_table_categorical("cohab.child.cat"), "cohab.child.cat", "Children in the household", "number of children in the household?")

#### 2.2.3. Children under 5 in the household.

In [42]:
create_table_categorical("num.under5.cat")

,segment,num.under5.cat,percentage
1,R2,0,0.207885
2,R3.1,0,1.206825
3,R3.2,0,2.044410
4,R4,0,0.983094
5,U1,0,45.700500
6,U2,0,50.272702
7,U4,0,19.211581
8,R2,1,78.215492
9,R3.1,1,64.722992
10,R3.2,1,72.544963


In [43]:
create_figure_stacked_bars(create_table_categorical("num.under5.cat"), "num.under5.cat", "Number of children under 5 in the household", "number of children under 5?")

#### 2.2.4. Number of adults in the household.

In [44]:
create_table_categorical("cohab.adult.sum.grp")

,segment,cohab.adult.sum.grp,percentage
1,R2,1-2,1.318636
2,R3.1,1-2,2.056228
3,R3.2,1-2,1.008674
4,R4,1-2,4.756066
5,U1,1-2,1.963777
6,U2,1-2,0.000000
7,U4,1-2,4.714094
8,R2,2-4,83.689846
9,R3.1,2-4,68.788232
10,R3.2,2-4,84.562552


In [45]:
create_figure_stacked_bars(create_table_categorical("cohab.adult.sum.grp"), "cohab.adult.sum.grp", "Number of adults in the household", "number of adults?")

#### 2.2.5. The respondent has ever had a child.

In [46]:
create_table("child.ever", True)

,segment,child.ever,percentage
1,R2,Yes,100.0
2,R3.1,Yes,100.0
3,R3.2,Yes,100.0
4,R4,Yes,100.0
5,U1,Yes,100.0
6,U2,Yes,100.0
7,U4,Yes,100.0


In [47]:
create_figure_stacked_bars(create_table("child.ever", True), "child.ever", "The respondent has ever had a child", "respondent has ever had a child?")

#### 2.2.6. The household has access to a bank account.

In [48]:
create_table("hh.bank", True)

,segment,hh.bank,percentage
1,R2,No,5.517026
2,R3.1,No,5.461910
3,R3.2,No,8.164224
4,R4,No,15.759410
5,U1,No,3.578982
6,U2,No,2.278106
7,U4,No,7.888894
8,R2,Yes,94.482974
9,R3.1,Yes,94.538090
10,R3.2,Yes,91.835776


In [49]:
create_figure_stacked_bars(create_table("hh.bank", True), "hh.bank", "The household has access to a bank account", "access to a bank account?")

#### 2.2.7. The house is owned by a male.

In [50]:
create_table("hh.house.owner.male", True)

,segment,hh.house.owner.male,percentage
1,R2,No,67.989335
2,R3.1,No,61.368593
3,R3.2,No,45.591847
4,R4,No,65.059554
5,U1,No,73.410413
6,U2,No,50.953941
7,U4,No,66.402457
8,R2,Yes,32.010665
9,R3.1,Yes,38.631407
10,R3.2,Yes,54.408153


In [51]:
create_figure_stacked_bars(create_table("hh.house.owner.male", True), "hh.house.owner.male", "The house is owned by a male", "the house is owned by a male?")

#### 2.2.8. Total number of household members.

In [52]:
create_table_categorical("hh.members.cat")

,segment,hh.members.cat,percentage
1,R2,1,62.445173
2,R3.1,1,41.896312
3,R3.2,1,63.361366
4,R4,1,23.614447
5,U1,1,60.193578
6,U2,1,32.815673
7,U4,1,35.329404
8,R2,2,19.623864
9,R3.1,2,23.107736
10,R3.2,2,18.940624


In [53]:
create_figure_stacked_bars(create_table_categorical("hh.members.cat"), "hh.members.cat", "Total number of household members", "members of the household?")

#### 2.2.9. The respondent has been married for 10 years or more.

In [54]:
create_table("married.10years", True)

,segment,married.10years,percentage
1,R2,No,76.266434
2,R3.1,No,76.119828
3,R3.2,No,81.540539
4,R4,No,51.787110
5,U1,No,46.765765
6,U2,No,36.888021
7,U4,No,28.285680
8,R2,Yes,23.733566
9,R3.1,Yes,23.880172
10,R3.2,Yes,18.459461


In [55]:
create_figure_stacked_bars(create_table("married.10years", True), "married.10years", "The respondent has been married for 10 years or more", "has been married for +10 years?")

#### 2.2.10. The woman needs permission to go to the hospital.

In [56]:
create_table_categorical("medical.permit")

,segment,medical.permit,percentage
1,R2,big problem,36.345657
2,R3.1,big problem,3.875953
3,R3.2,big problem,5.879819
4,R4,big problem,8.591115
5,U1,big problem,1.456251
6,U2,big problem,0.000000
7,U4,big problem,12.330470
8,R2,no problem,37.873261
9,R3.1,no problem,65.361921
10,R3.2,no problem,42.498804


In [57]:
create_figure_stacked_bars(create_table_categorical("medical.permit"), "medical.permit", "The woman needs permission to go to the hospital", "is it a problem?")

#### 2.2.11. The partner is absent from the household.

In [58]:
create_table_categorical("partner.absent")

,segment,partner.absent,percentage
1,R2,No,98.625019
2,R3.1,No,95.399113
3,R3.2,No,100.000000
4,R4,No,97.042558
5,U1,No,97.390432
6,U2,No,94.657936
7,U4,No,95.713426
8,R2,Yes,1.374981
9,R3.1,Yes,4.600887
10,R3.2,Yes,0.000000


In [59]:
create_figure_stacked_bars(create_table_categorical("partner.absent"), "partner.absent", "The partner is absent from the household", "partner absent?")

### 2.3. Household economics


#### 2.3.1. Bicycle.


In [60]:
create_table("hh.bike", True)

,segment,hh.bike,percentage
1,R2,No,62.762344
2,R3.1,No,83.475236
3,R3.2,No,64.384175
4,R4,No,98.545158
5,U1,No,85.536406
6,U2,No,81.112068
7,U4,No,88.779325
8,R2,Yes,37.237656
9,R3.1,Yes,16.524764
10,R3.2,Yes,35.615825


In [61]:
create_figure_stacked_bars(create_table("hh.bike", True), "hh.bike", "The household has a bicycle", "the household has a bicycle?").show()

#### 2.3.2. Cooking inside the house.


In [62]:
create_table("hh.cook.inside", True)

,segment,hh.cook.inside,percentage
1,R2,No,72.794838
2,R3.1,No,32.889235
3,R3.2,No,65.210978
4,R4,No,14.401789
5,U1,No,26.916575
6,U2,No,13.175048
7,U4,No,25.814281
8,R2,Yes,27.205162
9,R3.1,Yes,67.110765
10,R3.2,Yes,34.789022


In [63]:
create_figure_stacked_bars(create_table("hh.cook.inside", True), "hh.cook.inside", "The household cooks inside the house", "the household cooks inside the house?").show()

#### 2.3.3. Wealth index based on household assets.


In [64]:
create_table_categorical("assets")

,segment,assets,percentage
1,R2,0,2.466706
2,R3.1,0,0.000000
3,R3.2,0,14.199430
4,R4,0,16.015608
5,U1,0,0.454913
...,...,...,...
59,R3.2,8,0.000000
60,R4,8,0.000000
61,U1,8,0.000000
62,U2,8,0.906072


In [65]:
create_figure_stacked_bars(create_table_categorical("assets"), "assets", "Wealth index based on household assets", "index")

#### 2.3.4. Motorcycle or scooter.


In [66]:
create_table("hh.motor", True)

,segment,hh.motor,percentage
1,R2,No,62.257088
2,R3.1,No,67.137683
3,R3.2,No,90.512806
4,R4,No,99.559600
5,U1,No,47.550616
6,U2,No,57.342396
7,U4,No,90.131203
8,R2,Yes,37.742912
9,R3.1,Yes,32.862317
10,R3.2,Yes,9.487194


In [67]:
create_figure_stacked_bars(create_table("hh.motor", True), "hh.motor", "The household has a motorcycle or scooter", "the household has a motorcycle or scooter").show()

#### 2.3.5. Transportation barriers to accessing medical care.


In [68]:
create_table("transport", True)

,segment,transport,percentage
1,R2,No,37.672982
2,R3.1,No,46.054370
3,R3.2,No,55.389767
4,R4,No,96.635428
5,U1,No,35.636626
6,U2,No,36.984343
7,U4,No,76.387241
8,R2,Yes,62.327018
9,R3.1,Yes,53.945630
10,R3.2,Yes,44.610233


In [69]:
create_figure_stacked_bars(create_table("transport", True), "transport", "The woman faces transportation barriers to accessing medical care", "the woman faces transportation barriers to accessing medical care").show()

#### 2.3.6. Wealth index based on urban/rural residence.


In [70]:
create_table_categorical("wealth.index.ur")

,segment,wealth.index.ur,percentage
1,R2,1: poorest,7.390154
2,R3.1,1: poorest,0.000000
3,R3.2,1: poorest,41.678406
4,R4,1: poorest,43.889343
5,U1,1: poorest,33.800560
6,U2,1: poorest,32.500841
7,U4,1: poorest,94.934570
8,R2,2: poorer,35.177678
9,R3.1,2: poorer,13.211324
10,R3.2,2: poorer,49.890684


In [71]:
create_figure_stacked_bars(create_table_categorical("wealth.index.ur"), "wealth.index.ur", "Wealth index based on urban/rural residence", "Index")

#### 2.3.7. Area of land owned by the household in hectares.

In [72]:
create_table_categorical("hh.land.ha")

,segment,hh.land.ha,percentage
1,R2,0,60.257616
2,R3.1,0,81.128071
3,R3.2,0,89.219340
4,R4,0,81.505032
5,U1,0,92.032020
6,U2,0,91.078775
7,U4,0,86.376920
8,R2,1: 0.1-0.5 ha,1.212745
9,R3.1,1: 0.1-0.5 ha,5.351683
10,R3.2,1: 0.1-0.5 ha,2.555137


In [73]:
create_figure_stacked_bars(create_table_categorical("hh.land.ha"), "hh.land.ha", "Area of land owned by the household in hectares", "area of land in hectares?")

#### 2.3.8. The household has only one bedroom.

In [74]:
create_table("bedroom.slum", True)

,segment,bedroom.slum,percentage
1,R2,No,84.431880
2,R3.1,No,86.170384
3,R3.2,No,77.079002
4,R4,No,53.074975
5,U1,No,89.537005
6,U2,No,88.531961
7,U4,No,58.374099
8,R2,Yes,15.568120
9,R3.1,Yes,13.829616
10,R3.2,Yes,22.920998


In [75]:
create_figure_stacked_bars(create_table("bedroom.slum", True), "bedroom.slum", "The household has only one bedroom", "only one bedroom?")

#### 2.3.9. The household has electricity.

In [76]:
create_table("hh.electricity", True)

,segment,hh.electricity,percentage
1,R2,No,1.914180
2,R3.1,No,1.319897
3,R3.2,No,7.940274
4,R4,No,16.447450
5,U1,No,0.000000
6,U2,No,0.600165
7,U4,No,18.469457
8,R2,Yes,98.085820
9,R3.1,Yes,98.680103
10,R3.2,Yes,92.059726


In [77]:
create_figure_stacked_bars(create_table("hh.electricity", True), "hh.electricity", "The household has electricity", "the household has electricity?")

#### 2.3.10. The household has access to the internet.

In [78]:
create_table("hh.internet", True)

,segment,hh.internet,percentage
1,R2,No,41.846681
2,R3.1,No,31.012439
3,R3.2,No,85.145913
4,R4,No,79.751427
5,U1,No,38.714686
6,U2,No,27.360551
7,U4,No,55.321223
8,R2,Yes,58.153319
9,R3.1,Yes,68.987561
10,R3.2,Yes,14.854087


In [79]:
create_figure_stacked_bars(create_table("hh.internet", True), "hh.internet", "Household has access to the internet", "access to the internet?")

#### 2.3.11. The household has a refrigerator.

In [80]:
create_table("hh.refrig", True)

,segment,hh.refrig,percentage
1,R2,No,93.608941
2,R3.1,No,83.638816
3,R3.2,No,99.799578
4,R4,No,99.902698
5,U1,No,56.681880
6,U2,No,48.094744
7,U4,No,98.194123
8,R2,Yes,6.391059
9,R3.1,Yes,16.361184
10,R3.2,Yes,0.200422


In [81]:
create_figure_stacked_bars(create_table("hh.refrig", True), "hh.refrig", "The household has a refrigerator", "the household has a refrigerator?")

#### 2.3.12. The woman faces financial barriers to accessing medical care.

In [82]:
create_table_categorical("medical.money")

,segment,medical.money,percentage
1,R2,big problem,81.451682
2,R3.1,big problem,11.987061
3,R3.2,big problem,30.541168
4,R4,big problem,47.444842
5,U1,big problem,48.006470
6,U2,big problem,3.953201
7,U4,big problem,82.656405
8,R2,no problem,1.610237
9,R3.1,no problem,34.003022
10,R3.2,no problem,17.288594


In [83]:
create_figure_stacked_bars(create_table_categorical("medical.money"), "medical.money", "The woman faces financial barriers to accessing medical care", "is it a problem?")

### 2.4. mental and healthcare models


#### 2.4.1. Drinking water.


In [84]:
create_table("hh.water.purify", True)

,segment,hh.water.purify,percentage
1,R2,No,51.865586
2,R3.1,No,22.899656
3,R3.2,No,61.733854
4,R4,No,11.928366
5,U1,No,15.650124
6,U2,No,4.309618
7,U4,No,16.578198
8,R2,Yes,48.134414
9,R3.1,Yes,77.100344
10,R3.2,Yes,38.266146


In [85]:
create_figure_stacked_bars(create_table("hh.water.purify", True), "hh.water.purify", "The household purifies drinking water", "the household purifies drinking water?").show()

#### 2.4.2. Family planning through media.


In [86]:
create_table("heard.FP.media", True)

,segment,heard.FP.media,percentage
1,R2,No,22.481850
2,R3.1,No,19.934490
3,R3.2,No,44.214339
4,R4,No,28.370983
5,U1,No,17.617859
6,U2,No,13.095306
7,U4,No,21.820734
8,R2,Yes,77.518150
9,R3.1,Yes,80.065510
10,R3.2,Yes,55.785661


In [87]:
create_figure_stacked_bars(create_table("heard.FP.media", True), "heard.FP.media", "The woman has heard of family planning through media", "the woman has heard of family planning through media?").show()

#### 2.4.3. Barriers to accessing a female medical provider.


In [88]:
create_table_categorical("medical.female.provider")

,segment,medical.female.provider,percentage
1,R2,big problem,86.658779
2,R3.1,big problem,7.624203
3,R3.2,big problem,3.746391
4,R4,big problem,30.857325
5,U1,big problem,28.395619
6,U2,big problem,7.626503
7,U4,big problem,47.024204
8,R2,no problem,0.629964
9,R3.1,no problem,44.582010
10,R3.2,no problem,25.217148


In [89]:
create_figure_stacked_bars(create_table_categorical("medical.female.provider"), "medical.female.provider", "The woman faces barriers to accessing a female medical provider", "is it a big problem?")

#### 2.4.4. Distance barriers to accessing medical care.


In [90]:
create_table_categorical("medical.distance")

,segment,medical.distance,percentage
1,R2,big problem,86.304690
2,R3.1,big problem,11.454256
3,R3.2,big problem,30.125796
4,R4,big problem,44.941170
5,U1,big problem,37.876388
6,U2,big problem,0.223396
7,U4,big problem,66.693617
8,R2,no problem,0.959246
9,R3.1,no problem,36.014558
10,R3.2,no problem,12.210336


In [91]:
create_figure_stacked_bars(create_table_categorical("medical.distance"), "medical.distance", "The woman faces distance barriers to accessing medical care", "is it a big problem?")

#### 2.4.5. The woman faces barriers due to lack of medical providers.

In [92]:
create_table_categorical("medical.no.provider")

,segment,medical.no.provider,percentage
1,R2,big problem,94.912852
2,R3.1,big problem,20.910893
3,R3.2,big problem,18.165502
4,R4,big problem,54.162684
5,U1,big problem,50.866205
6,U2,big problem,24.502038
7,U4,big problem,65.073246
8,R2,no problem,0.000000
9,R3.1,no problem,26.877260
10,R3.2,no problem,13.271107


In [93]:
create_figure_stacked_bars(create_table_categorical("medical.no.provider"), "medical.no.provider", "The woman faces barriers due to lack of medical providers", "is it a problem?")

#### 2.4.6. The woman faces barriers due to lack of medical drugs.

In [94]:
create_table_categorical("medical.no.drug")

,segment,medical.no.drug,percentage
1,R2,big problem,96.510062
2,R3.1,big problem,31.579095
3,R3.2,big problem,26.557784
4,R4,big problem,62.993152
5,U1,big problem,61.190041
6,U2,big problem,35.695413
7,U4,big problem,80.208381
8,R2,no problem,0.000000
9,R3.1,no problem,26.289990
10,R3.2,no problem,10.423903


In [95]:
create_figure_stacked_bars(create_table_categorical("medical.no.drug"), "medical.no.drug", "The woman faces barriers due to lack of medical drugs", "is it a problem?")

#### 2.4.7. The woman has misinformation about TB transmission. 

In [96]:
create_table("know.tb.misinfo", True)

,segment,know.tb.misinfo,percentage
1,R2,No,56.253502
2,R3.1,No,49.985899
3,R3.2,No,87.771662
4,R4,No,42.587522
5,U1,No,28.107340
6,U2,No,27.619035
7,U4,No,40.041880
8,R2,Yes,43.746498
9,R3.1,Yes,50.014101
10,R3.2,Yes,12.228338


In [97]:
create_figure_stacked_bars(create_table("know.tb.misinfo", True), "know.tb.misinfo", "The woman has misinformation about TB transmission", "misinformation about TB transmission?")

#### 2.4.8. The woman faces barriers to accessing medical care alone.

In [98]:
create_table_categorical("medical.alone")

,segment,medical.alone,percentage
1,R2,big problem,54.667565
2,R3.1,big problem,2.750502
3,R3.2,big problem,6.446079
4,R4,big problem,23.040176
5,U1,big problem,12.756852
6,U2,big problem,0.000000
7,U4,big problem,49.040996
8,R2,no problem,24.218965
9,R3.1,no problem,48.811783
10,R3.2,no problem,37.286909


In [99]:
create_figure_stacked_bars(create_table_categorical("medical.alone"), "medical.alone", "The woman faces barriers to accessing medical care alone", "is it a problem?")

#### 2.4.9. The woman knows TB spreads through coughing.

In [100]:
create_table("know.tb.cough", True)

,segment,know.tb.cough,percentage
1,R2,No,24.781681
2,R3.1,No,35.965762
3,R3.2,No,70.617380
4,R4,No,46.993753
5,U1,No,9.208414
6,U2,No,22.529890
7,U4,No,53.057502
8,R2,Yes,75.218319
9,R3.1,Yes,64.034238
10,R3.2,Yes,29.382620


In [101]:
create_figure_stacked_bars(create_table("know.tb.cough", True), "know.tb.cough", "The woman knows TB spreads through coughing", "the woman knows TB spreads through coughing?")

#### 2.4.10. The woman knows TB is curable.

In [102]:
create_table("know.tb.cure", True)

,segment,know.tb.cure,percentage
1,R2,No,13.536890
2,R3.1,No,9.287639
3,R3.2,No,13.010653
4,R4,No,19.385930
5,U1,No,32.514000
6,U2,No,9.324854
7,U4,No,9.322534
8,R2,Yes,86.463110
9,R3.1,Yes,90.712361
10,R3.2,Yes,86.989347


In [103]:
create_figure_stacked_bars(create_table("know.tb.cure", True), "know.tb.cure", "The woman knows TB is curable", "the woman knows TB is curable?")

#### 2.4.11. The woman has health insurance.

In [104]:
create_table("health.insurance", True)

,segment,health.insurance,percentage
1,R2,No,26.090042
2,R3.1,No,30.058338
3,R3.2,No,41.597070
4,R4,No,28.990739
5,U1,No,28.456916
6,U2,No,43.017111
7,U4,No,55.316940
8,R2,Yes,73.909958
9,R3.1,Yes,69.941662
10,R3.2,Yes,58.402930


In [105]:
create_figure_stacked_bars(create_table("health.insurance", True), "health.insurance", "The woman has health insurance", "the woman has health insurance?")

#### 2.4.12. The woman faces transportation barriers to accessing medical care.

In [106]:
create_table_categorical("medical.transport")

,segment,medical.transport,percentage
1,R2,big problem,86.316838
2,R3.1,big problem,5.467209
3,R3.2,big problem,23.427243
4,R4,big problem,44.118313
5,U1,big problem,31.284312
6,U2,big problem,0.000000
7,U4,big problem,51.277980
8,R2,no problem,1.458375
9,R3.1,no problem,40.440372
10,R3.2,no problem,14.035016


In [107]:
create_figure_stacked_bars(create_table_categorical("medical.transport"), "medical.transport", " The woman faces transportation barriers to accessing medical care", "is it a problem?")

### 2.5. Life in the community


#### 2.5.1. Visits by a CHW in the last 3 months.


In [108]:
create_table("chw.3mo", True)

,segment,chw.3mo,percentage
1,R2,No,60.694581
2,R3.1,No,57.779191
3,R3.2,No,78.558627
4,R4,No,45.748687
5,U1,No,81.496057
6,U2,No,85.506693
7,U4,No,65.763515
8,R2,Yes,39.305419
9,R3.1,Yes,42.220809
10,R3.2,Yes,21.441373


In [109]:
create_figure_stacked_bars(create_table("chw.3mo", True), "chw.3mo", "The woman has been visited by a CHW in the last 3 months", "the woman has been visited by a CHW in the last 3 months?").show()

#### 2.5.2. Family planning information from ICDS.


In [110]:
create_table("source.fp.icds", True)

,segment,source.fp.icds,percentage
1,R2,No,97.779092
2,R3.1,No,96.814426
3,R3.2,No,98.645223
4,R4,No,98.329363
5,U1,No,97.866366
6,U2,No,99.754128
7,U4,No,98.985628
8,R2,Yes,2.220908
9,R3.1,Yes,3.185574
10,R3.2,Yes,1.354777


In [111]:
create_figure_stacked_bars(create_table("source.fp.icds", True), "source.fp.icds", "The woman gets family planning information from ICDS", "the woman gets family planning information from ICDS?").show()

#### 2.5.3. The woman gets family planning information from CHW.

In [112]:
create_table("source.fp.chw", True)

,segment,source.fp.chw,percentage
1,R2,No,78.262470
2,R3.1,No,85.498851
3,R3.2,No,89.453952
4,R4,No,90.373835
5,U1,No,94.321617
6,U2,No,98.500066
7,U4,No,95.744301
8,R2,Yes,21.737530
9,R3.1,Yes,14.501149
10,R3.2,Yes,10.546048


In [113]:
create_figure_stacked_bars(create_table("source.fp.chw", True), "source.fp.chw", "The woman gets family planning information from CHW", "the woman gets family planning information from CHW?")

#### 2.5.4. The woman received prenatal care from a CHW.

In [114]:
create_table("chw.prenatal", True)

,segment,chw.prenatal,percentage
1,R2,No,79.238759
2,R3.1,No,88.794013
3,R3.2,No,88.967306
4,R4,No,94.102303
5,U1,No,96.467420
6,U2,No,95.585966
7,U4,No,94.781400
8,R2,Yes,20.761241
9,R3.1,Yes,11.205987
10,R3.2,Yes,11.032694


In [115]:
create_figure_stacked_bars(create_table("chw.prenatal", True), "chw.prenatal", "The woman received prenatal care from a CHW", "the woman received prenatal care from a CHW?")

### 2.6. Larger environment

#### 2.6.1. The woman is exposed to media.


In [116]:
create_table("media", True)

,segment,media,percentage
1,R2,No,21.849224
2,R3.1,No,3.389059
3,R3.2,No,26.931150
4,R4,No,38.846690
5,U1,No,2.271468
6,U2,No,4.361110
7,U4,No,24.424884
8,R2,Yes,78.150776
9,R3.1,Yes,96.610941
10,R3.2,Yes,73.068850


In [117]:
create_figure_stacked_bars(create_table("media", True), "media", "The woman is exposed to media", "the woman is exposed to media?").show()

#### 2.6.2. How frequently the woman watches TV.

In [118]:
create_table_categorical("tv.freq")

,segment,tv.freq,percentage
1,R2,at least once a week,54.845290
2,R3.1,at least once a week,55.228926
3,R3.2,at least once a week,49.838614
4,R4,at least once a week,19.001324
5,U1,at least once a week,63.707986
6,U2,at least once a week,76.168993
7,U4,at least once a week,45.000193
8,R2,less than once a week,21.405088
9,R3.1,less than once a week,31.857951
10,R3.2,less than once a week,21.406838


In [119]:
create_figure_stacked_bars(create_table_categorical("tv.freq"), "tv.freq", "How frequently the woman watches TV", "how frequently?")

#### 2.6.3. The household has access to improved sanitation facilities.

In [120]:
create_table("hh.WASH.toilet", True)

,segment,hh.WASH.toilet,percentage
1,R2,No,43.209109
2,R3.1,No,34.333945
3,R3.2,No,78.814337
4,R4,No,44.728639
5,U1,No,24.073291
6,U2,No,21.924858
7,U4,No,34.560353
8,R2,Yes,56.790891
9,R3.1,Yes,65.666055
10,R3.2,Yes,21.185663


In [121]:
create_figure_stacked_bars(create_table("hh.WASH.toilet", True), "hh.WASH.toilet", "the household has access to improved sanitation facilities", "access to improved sanitation facilities?")

#### 2.6.4. The household has a dirt floor.

In [122]:
create_table("floor.slum", True)

,segment,floor.slum,percentage
1,R2,No,61.297518
2,R3.1,No,90.655735
3,R3.2,No,30.001554
4,R4,No,54.856655
5,U1,No,94.959405
6,U2,No,91.251209
7,U4,No,53.550629
8,R2,Yes,38.702482
9,R3.1,Yes,9.344265
10,R3.2,Yes,69.998446


In [123]:
create_figure_stacked_bars(create_table("floor.slum", True), "floor.slum", "The household has a dirt floor", "dirt floor?")

#### 2.6.5. Indicates how frequently the woman reads newspapers or magazines.

In [124]:
create_table_categorical("np.mag.freq")

,segment,np.mag.freq,percentage
1,R2,at least once a week,1.127336
2,R3.1,at least once a week,19.842828
3,R3.2,at least once a week,6.088561
4,R4,at least once a week,10.940004
5,U1,at least once a week,9.107814
6,U2,at least once a week,44.944905
7,U4,at least once a week,5.724396
8,R2,less than once a week,15.999751
9,R3.1,less than once a week,44.269023
10,R3.2,less than once a week,11.578443


In [125]:
create_figure_stacked_bars(create_table_categorical("np.mag.freq"), "np.mag.freq", "how frequently the woman reads newspapers or magazines", "how frequently the woman reads newspapers or magazines?")

#### 2.6.6. The household has access to piped water.

In [126]:
create_table("hh.pipe.water", True)

,segment,hh.pipe.water,percentage
1,R2,No,59.279132
2,R3.1,No,53.143164
3,R3.2,No,85.729156
4,R4,No,61.179314
5,U1,No,14.824301
6,U2,No,20.658387
7,U4,No,51.370244
8,R2,Yes,40.720868
9,R3.1,Yes,46.856836
10,R3.2,Yes,14.270844


In [127]:
create_figure_stacked_bars(create_table("hh.pipe.water", True), "hh.pipe.water", "the household has access to piped water", "the household has access to piped water?")

#### 2.6.7. The household has a latrine.

In [128]:
create_table("latrine", True)

,segment,latrine,percentage
1,R2,No,86.310299
2,R3.1,No,92.575809
3,R3.2,No,60.239400
4,R4,No,90.214131
5,U1,No,98.726774
6,U2,No,97.010347
7,U4,No,91.197958
8,R2,Yes,13.689701
9,R3.1,Yes,7.424191
10,R3.2,Yes,39.760600


In [129]:
create_figure_stacked_bars(create_table("latrine", True), "latrine", "The household has a latrine", "the household has a latrine?")

#### 2.6.8. The household uses a cooking fuel that does not produce smoke.

In [130]:
create_table("hh.cook.fuel.nosmoke", True)

,segment,hh.cook.fuel.nosmoke,percentage
1,R2,No,73.481643
2,R3.1,No,47.569627
3,R3.2,No,95.129147
4,R4,No,95.099835
5,U1,No,22.271261
6,U2,No,25.066412
7,U4,No,73.063972
8,R2,Yes,26.518357
9,R3.1,Yes,52.430373
10,R3.2,Yes,4.870853


In [131]:
create_figure_stacked_bars(create_table("hh.cook.fuel.nosmoke", True), "hh.cook.fuel.nosmoke", "The household uses a cooking fuel that does not produce smoke", "using a cooking fuel that does not produce smoke?")

#### 2.6.9. Indicates the number of bedrooms in the household.

In [132]:
create_table_categorical("hh.bedrooms")

,segment,hh.bedrooms,percentage
1,R2,0-1,20.247089
2,R3.1,0-1,20.344031
3,R3.2,0-1,41.700699
4,R4,0-1,35.544920
5,U1,0-1,12.996931
6,U2,0-1,9.002674
7,U4,0-1,30.661633
8,R2,2,44.509462
9,R3.1,2,32.452549
10,R3.2,2,38.769476


In [133]:
create_figure_stacked_bars(create_table_categorical("hh.bedrooms"), "hh.bedrooms", "Number of bedrooms in the household", "number of bedrooms?")

#### 2.6.10. Indicates how frequently the woman listens to the radio.

In [134]:
create_table_categorical("radio.freq")

,segment,radio.freq,percentage
1,R2,at least once a week,0.239101
2,R3.1,at least once a week,8.226134
3,R3.2,at least once a week,2.574701
4,R4,at least once a week,4.307476
5,U1,at least once a week,0.261211
6,U2,at least once a week,14.064340
7,U4,at least once a week,0.000000
8,R2,less than once a week,6.993177
9,R3.1,less than once a week,27.345254
10,R3.2,less than once a week,4.414115


In [135]:
create_figure_stacked_bars(create_table_categorical("radio.freq"), "radio.freq", "How frequently the woman listens to the radio", "how frequently the woman listens to the radio?")

## 3. Outcome variables

### 3.1. ANC/PNC

#### 3.1.1 Number of ANC visits

When the number of ANC visits is 4 or more (0 = yes, 1 = no)

In [136]:
create_table('anc.4plus')

,segment,anc.4plus,percentage
1,R2,Yes,33.954403
2,R3.1,Yes,64.075674
3,R3.2,Yes,39.127489
4,R4,Yes,61.531026
5,U1,Yes,85.245235
6,U2,Yes,89.266721
7,U4,Yes,69.360886
8,R2,No,66.045597
9,R3.1,No,35.924326
10,R3.2,No,60.872511


In [137]:
create_figure_stacked_bars(create_table('anc.4plus'), 'anc.4plus', 'Number of ANC visits', 'number of ANC visits is 4 or more?').show()

#### 3.1.2. Woman had a health check after birth

In [138]:
create_table("wom.hlthck")


,segment,wom.hlthck,percentage
1,R2,Yes,89.583772
2,R3.1,Yes,82.150593
3,R3.2,Yes,67.580770
4,R4,Yes,78.500463
5,U1,Yes,91.439474
6,U2,Yes,91.258211
7,U4,Yes,68.333033
8,R2,No,10.416228
9,R3.1,No,17.849407
10,R3.2,No,32.419230


In [139]:
create_figure_stacked_bars(create_table("wom.hlthck"), "wom.hlthck", "Health check after birth, by segment", "Woman had Health Check?").show()

#### 3.1.3 Baby health check after birth

In [140]:
create_table("baby.hlthck")

,segment,baby.hlthck,percentage
1,R2,Yes,84.273729
2,R3.1,Yes,80.123396
3,R3.2,Yes,57.904065
4,R4,Yes,69.721578
5,U1,Yes,85.040613
6,U2,Yes,95.652150
7,U4,Yes,62.979796
8,R2,No,15.726271
9,R3.1,No,19.876604
10,R3.2,No,42.095935


In [141]:
create_figure_stacked_bars(create_table("baby.hlthck"), "baby.hlthck", "Baby health check after birth", "baby had a health check after birth?")

### 3.2. Breastfeeding

#### 3.2.1. Latest child was ever breastfed

In [142]:
create_table("breastfed")

,segment,breastfed,percentage
1,R2,Yes,63.388845
2,R3.1,Yes,55.850335
3,R3.2,Yes,68.322058
4,R4,Yes,58.275371
5,U1,Yes,37.572293
6,U2,Yes,42.368074
7,U4,Yes,55.122844
8,R2,No,36.611155
9,R3.1,No,44.149665
10,R3.2,No,31.677942


In [143]:
create_figure_stacked_bars(create_table("breastfed"), "breastfed", "Latest child was ever breastfed", "latest child was ever breastfed?").show()

### 3.3.Home Birth

#### 3.3.1. Latest birth was a home birth

In [144]:
create_table("hb.1", True)

,segment,hb.1,percentage
1,R2,No,89.424051
2,R3.1,No,82.266008
3,R3.2,No,69.519908
4,R4,No,53.766674
5,U1,No,100.000000
6,U2,No,94.776239
7,U4,No,66.897517
8,R2,Yes,10.575949
9,R3.1,Yes,17.733992
10,R3.2,Yes,30.480092


In [145]:
create_figure_stacked_bars(create_table("hb.1", True), "hb.1", "Latest birth was a home birth", "latest birth was a home birth?", True).show()

### 3.4. Family Planning

#### 3.4.1. Using modern family planning methods

In [146]:
create_table("fp.mod.now")

,segment,fp.mod.now,percentage
1,R2,Yes,33.769018
2,R3.1,Yes,26.214416
3,R3.2,Yes,28.557600
4,R4,Yes,14.641182
5,U1,Yes,45.745276
6,U2,Yes,15.512548
7,U4,Yes,24.283048
8,R2,No,66.230982
9,R3.1,No,73.785584
10,R3.2,No,71.442400


In [147]:
create_figure_stacked_bars(create_table("fp.mod.now"), "fp.mod.now", "Respondent is currently using modern family planning methods", "currently using modern family planning methods?").show()

### 3.5. Malnourishment


#### 3.5.3. Child is underweight


In [148]:
create_table("undwgt.yn", True)

,segment,undwgt.yn,percentage
1,R2,No,69.442223
2,R3.1,No,70.013645
3,R3.2,No,80.725611
4,R4,No,56.073651
5,U1,No,87.985050
6,U2,No,80.101975
7,U4,No,43.000825
8,R2,Yes,30.557777
9,R3.1,Yes,29.986355
10,R3.2,Yes,19.274389


In [149]:
create_figure_stacked_bars(create_table("undwgt.yn", True), "undwgt.yn", "Child is underweight", "child is underweight?", True).show()

#### 3.5.4. Child is stunted


In [150]:
create_table("stunt.cat2.yn", True)

,segment,stunt.cat2.yn,percentage
1,R2,No,71.776346
2,R3.1,No,64.632660
3,R3.2,No,70.688703
4,R4,No,49.407831
5,U1,No,87.167343
6,U2,No,82.665526
7,U4,No,37.318638
8,R2,Yes,28.223654
9,R3.1,Yes,35.367340
10,R3.2,Yes,29.311297


In [151]:
create_figure_stacked_bars(create_table("stunt.cat2.yn", True), "stunt.cat2.yn", "Child is stunted", "child is stunted?", True).show()

#### 3.5.5. Child is wasted

In [152]:
create_table("waste.cat2.yn", True)

,segment,waste.cat2.yn,percentage
1,R2,No,79.400911
2,R3.1,No,87.353331
3,R3.2,No,89.996686
4,R4,No,86.550265
5,U1,No,88.847283
6,U2,No,87.921458
7,U4,No,85.527868
8,R2,Yes,20.599089
9,R3.1,Yes,12.646669
10,R3.2,Yes,10.003314


In [153]:
create_figure_stacked_bars(create_table("waste.cat2.yn", True), "waste.cat2.yn", "Child is wasted", "child is wasted?", True).show()

### 3.6. Menstrual Hygeine


#### 3.6.1. Using any material to wash during menstruation

In [154]:
create_table("no.mens.wash", True)

,segment,no.mens.wash,percentage
1,R2,No,93.266036
2,R3.1,No,82.316614
3,R3.2,No,93.739297
4,R4,No,82.247334
5,U1,No,82.156934
6,U2,No,76.529520
7,U4,No,59.015405
8,R2,Yes,6.733964
9,R3.1,Yes,17.683386
10,R3.2,Yes,6.260703


In [155]:
create_figure_stacked_bars(create_table("no.mens.wash", True), "no.mens.wash", "Respondent did not use any material to wash during menstruation", "not using any material to wash during menstruation?", True).show()

### 3.7. Hyper Tension



#### 3.7.1. Respondent has hypertension

In [156]:
create_table("hypertension", True)

,segment,hypertension,percentage
1,R2,No,92.391819
2,R3.1,No,89.094793
3,R3.2,No,95.036938
4,R4,No,87.797447
5,U1,No,81.723592
6,U2,No,89.093030
7,U4,No,88.392217
8,R2,Yes,7.608181
9,R3.1,Yes,10.905207
10,R3.2,Yes,4.963062


In [157]:
create_figure_stacked_bars(create_table("hypertension", True), "hypertension", "Respondent has hypertension based on systolic and diastolic blood pressure", "respondent has hypertension?", True).show()

### 3.8. Blood Glucose



#### 3.8.1. Respondent has high blood glucose levels

In [158]:
create_table("blood_glucose", True)

,segment,blood_glucose,percentage
1,R2,No,90.971593
2,R3.1,No,92.945898
3,R3.2,No,87.367330
4,R4,No,93.425324
5,U1,No,83.926294
6,U2,No,86.114568
7,U4,No,90.604383
8,R2,Yes,9.028407
9,R3.1,Yes,7.054102
10,R3.2,Yes,12.632670


In [159]:
create_figure_stacked_bars(create_table("blood_glucose", True), "blood_glucose", "Respondent has high blood glucose levels", "has high blood glucose levels?", True).show()

### 3.9. Child mortality



#### 3.9.1. Child died under the age of 1


In [160]:
create_table("u1mort.yn", True)

,segment,u1mort.yn,percentage
1,R2,No,99.301224
2,R3.1,No,96.798042
3,R3.2,No,95.662879
4,R4,No,88.104873
5,U1,No,97.923191
6,U2,No,95.361741
7,U4,No,78.847857
8,R2,Yes,0.698776
9,R3.1,Yes,3.201958
10,R3.2,Yes,4.337121


In [161]:
create_figure_stacked_bars(create_table("u1mort.yn", True), "u1mort.yn", "Child died under the age of 1", "child died under the age of 1?", True).show()

#### 3.9.2. Child died under the age of 5

In [162]:
create_table("u5mort.yn", True)

,segment,u5mort.yn,percentage
1,R2,No,99.301224
2,R3.1,No,96.319348
3,R3.2,No,94.985998
4,R4,No,84.936739
5,U1,No,97.512634
6,U2,No,95.361741
7,U4,No,76.371804
8,R2,Yes,0.698776
9,R3.1,Yes,3.680652
10,R3.2,Yes,5.014002


In [163]:
create_figure_stacked_bars(create_table("u5mort.yn", True), "u5mort.yn", "Child died under the age of 5", "child died under the age of 5?", True).show()

### 3.10. Anemia

#### 3.10.1. Anemia level

In [164]:
create_table_categorical("anemia.level")

,segment,anemia.level,percentage
1,R2,mild,29.235694
2,R3.1,mild,21.940050
3,R3.2,mild,23.190632
4,R4,mild,22.713764
5,U1,mild,28.814564
6,U2,mild,32.411285
7,U4,mild,28.113495
8,R2,moderate,33.109212
9,R3.1,moderate,21.118093
10,R3.2,moderate,26.424188


In [165]:
create_figure_stacked_bars(create_table_categorical("anemia.level"), "anemia.level", "Anemia level", "Anemia level")

### 3.11. BMI

#### 3.11.1 Woman's BMI

In [166]:
ro.r('median_bmi <- svyby(~w.bmi, ~segment, survey_design, svyquantile, quantiles = c(0.5), na.rm = TRUE)')
ro.r('median_bmi <- median_bmi %>% rename(median_bmi = w.bmi)')
ro.r('q1_bmi <- svyby(~w.bmi, ~segment, survey_design, svyquantile, quantiles = c(0.25), na.rm = TRUE)')
ro.r('q1_bmi <- q1_bmi %>%  rename(q1 = w.bmi)')
ro.r('q3_bmi <- svyby(~w.bmi, ~segment, survey_design, svyquantile, quantiles = c(0.75), na.rm = TRUE)')
ro.r('q3_bmi <- q3_bmi %>%  rename(q3 = w.bmi)')
ro.r('min_bmi <- svyby(~w.bmi, ~segment, survey_design, svyquantile, quantiles = c(0.0), na.rm = TRUE)')
ro.r('min_bmi <- min_bmi %>% rename(min = w.bmi)')
ro.r('max_bmi <- svyby(~w.bmi, ~segment, survey_design, svyquantile, quantiles = c(1.0), na.rm = TRUE)')
ro.r('max_bmi <- max_bmi %>% rename(max = w.bmi)')

ro.r('bmi_stats <- full_join(q1_bmi, q3_bmi, by = "segment")')
ro.r('bmi_stats <- full_join(bmi_stats, median_bmi, by = "segment")')
ro.r('bmi_stats <- full_join(bmi_stats, min_bmi, by = "segment")')
ro.r('bmi_stats <- full_join(bmi_stats, max_bmi, by = "segment")')
ro.r('bmi_stats <- bmi_stats %>% select("segment", "min", "q1", "median_bmi", "q3", "max")')

bmi_stats = ro.r('bmi_stats')
with (ro.default_converter + pandas2ri.converter).context():
        bmi_stats = ro.conversion.get_conversion().rpy2py(bmi_stats) 

print(bmi_stats)

  segment     min      q1  median_bmi      q3     max
1      R2  1609.0  2059.0      2152.0  2322.0  3082.0
2    R3.1  1495.0  2007.0      2170.0  2392.0  4340.0
3    R3.2  1573.0  2019.0      2120.0  2235.0  2987.0
4      R4  1553.0  1969.0      2125.0  2287.0  9998.0
5      U1  1763.0  2032.0      2169.0  2374.0  2910.0
6      U2  1581.0  2041.0      2257.0  2423.0  3258.0
7      U4  1751.0  2042.0      2198.0  2274.0  2962.0


In [167]:
# Create the box plot using the aggregated data
bmi_fig_2 = go.Figure()

for segment in bmi_stats['segment']:
    segment_data = bmi_stats[bmi_stats['segment'] == segment]
    bmi_fig_2.add_trace(go.Box(
        name=segment,
        x=[segment] * 5,
        q1=[segment_data['q1'].values[0]],
        median=[segment_data['median_bmi'].values[0]],
        q3=[segment_data['q3'].values[0]],
        lowerfence=[segment_data['min'].values[0]],
        upperfence=[segment_data['max'].values[0]]
    ))

# Update layout
bmi_fig_2.update_layout(
    title='BMI by Segment',
    xaxis_title='Segment',
    yaxis_title='BMI'
)

# Show the plot
bmi_fig_2.show()

### 3.12. Woman Stunting

#### 3.12.1. Woman Stunting Index

In [168]:
ro.r('median_si <- svyby(~w.haz, ~segment, survey_design, svyquantile, quantiles = c(0.5), na.rm = TRUE)')
ro.r('median_si <- median_si %>% rename(median_si = w.haz)')
ro.r('q1_si <- svyby(~w.haz, ~segment, survey_design, svyquantile, quantiles = c(0.25), na.rm = TRUE)')
ro.r('q1_si <- q1_si %>%  rename(q1 = w.haz)')
ro.r('q3_si <- svyby(~w.haz, ~segment, survey_design, svyquantile, quantiles = c(0.75), na.rm = TRUE)')
ro.r('q3_si <- q3_si %>%  rename(q3 = w.haz)')
ro.r('min_si <- svyby(~w.haz, ~segment, survey_design, svyquantile, quantiles = c(0.0), na.rm = TRUE)')
ro.r('min_si <- min_si %>% rename(min = w.haz)')
ro.r('max_si <- svyby(~w.haz, ~segment, survey_design, svyquantile, quantiles = c(1.0), na.rm = TRUE)')
ro.r('max_si <- max_si %>% rename(max = w.haz)')

ro.r('si_stats <- full_join(q1_si, q3_si, by = "segment")')
ro.r('si_stats <- full_join(si_stats, median_si, by = "segment")')
ro.r('si_stats <- full_join(si_stats, min_si, by = "segment")')
ro.r('si_stats <- full_join(si_stats, max_si, by = "segment")')
ro.r('si_stats <- si_stats %>% select("segment", "min", "q1", "median_si", "q3", "max")')

si_stats = ro.r('si_stats')
with (ro.default_converter + pandas2ri.converter).context():
        si_stats = ro.conversion.get_conversion().rpy2py(si_stats) 

print(si_stats)

  segment   min    q1  median_si    q3   max
1      R2 -4.88 -2.35      -1.83 -1.33  0.30
2    R3.1 -4.91 -2.85      -2.28 -1.73  0.69
3    R3.2 -4.19 -2.57      -2.06 -1.54  0.17
4      R4 -5.07 -3.14      -2.60 -2.08  0.65
5      U1 -4.34 -2.43      -1.83 -1.36 -0.59
6      U2 -4.44 -2.77      -2.25 -1.69  2.40
7      U4 -4.50 -3.24      -2.94 -2.01 -0.10


In [169]:
# Create the box plot using the aggregated data
si_fig_2 = go.Figure()

for segment in bmi_stats['segment']:
    segment_data = si_stats[si_stats['segment'] == segment]
    si_fig_2.add_trace(go.Box(
        name=segment,
        x=[segment] * 5,
        q1=[segment_data['q1'].values[0]],
        median=[segment_data['median_si'].values[0]],
        q3=[segment_data['q3'].values[0]],
        lowerfence=[segment_data['min'].values[0]],
        upperfence=[segment_data['max'].values[0]]
    ))

# Update layout
si_fig_2.update_layout(
    title='Woman Stunting Index by Segment',
    xaxis_title='Segment',
    yaxis_title='Stunting Index'
)

# Show the plot
si_fig_2.show()

#### 3.12.2. Woman Stunting Index

In [170]:
create_table_categorical("w.stunt.cat")

,segment,w.stunt.cat,percentage
1,R2,Moderately stunted,33.668644
2,R3.1,Moderately stunted,39.009189
3,R3.2,Moderately stunted,41.203195
4,R4,Moderately stunted,47.311069
5,U1,Moderately stunted,35.915563
6,U2,Moderately stunted,36.076641
7,U4,Moderately stunted,37.421961
8,R2,Not stunted,61.234756
9,R3.1,Not stunted,41.019503
10,R3.2,Not stunted,47.074898


In [171]:
create_figure_stacked_bars(create_table_categorical("w.stunt.cat"), "w.stunt.cat", "Woman stunting level", "Stunting level")

## 4. Deep Dive: Defining Variables

### 4.1. Woman and her past experience

#### 4.1.1 The respondent can read a whole sentence or part of it.

In [172]:
create_table_categorical("literacy")

,segment,literacy,percentage
1,R2,able to read only parts of sentence,25.660023
2,R3.1,able to read only parts of sentence,14.468502
3,R3.2,able to read only parts of sentence,27.868050
4,R4,able to read only parts of sentence,26.186493
5,U1,able to read only parts of sentence,18.823722
6,U2,able to read only parts of sentence,4.372787
7,U4,able to read only parts of sentence,39.064465
8,R2,able to read whole sentence,62.689207
9,R3.1,able to read whole sentence,82.785764
10,R3.2,able to read whole sentence,54.180575


In [173]:
create_figure_stacked_bars(create_table_categorical("literacy"), "literacy", "The respondent can read a whole sentence or part of it", "can read a whole sentence or part of it?")

#### 4.1.2 The respondent had sexual intercourse before age 18.

In [174]:
create_table("early.sex.18", True)

,segment,early.sex.18,percentage
1,R2,No,84.554430
2,R3.1,No,78.000572
3,R3.2,No,81.376889
4,R4,No,65.668392
5,U1,No,89.011313
6,U2,No,85.551173
7,U4,No,53.116542
8,R2,Yes,15.445570
9,R3.1,Yes,21.999428
10,R3.2,Yes,18.623111


In [175]:
create_figure_stacked_bars(create_table("early.sex.18", True), "early.sex.18", "The respondent had sexual intercourse before age 18", "had sex before 18?")

#### 4.1.3 The respondent started living with a partner before age 18.

In [176]:
create_table("early.cohab.18", True)

,segment,early.cohab.18,percentage
1,R2,No,84.140866
2,R3.1,No,76.182169
3,R3.2,No,76.664281
4,R4,No,63.340047
5,U1,No,88.311426
6,U2,No,83.842033
7,U4,No,43.359336
8,R2,Yes,15.859134
9,R3.1,Yes,23.817831
10,R3.2,Yes,23.335719


In [177]:
create_figure_stacked_bars(create_table("early.cohab.18", True), "early.cohab.18", "The respondent started living with a partner before age 18", "started living with a partner before age 18?")

#### 4.1.4 The respondent had their first birth before age 18.

In [178]:
create_table("early.birth.18", True)

,segment,early.birth.18,percentage
1,R2,No,93.045248
2,R3.1,No,90.250837
3,R3.2,No,91.025575
4,R4,No,80.113779
5,U1,No,92.898120
6,U2,No,91.414707
7,U4,No,72.907148
8,R2,Yes,6.954752
9,R3.1,Yes,9.749163
10,R3.2,Yes,8.974425


In [179]:
create_figure_stacked_bars(create_table("early.birth.18", True), "early.birth.18", "The respondent had their first birth before age 18", "first birth before age 18?")

#### 4.1.5. Age group at which the respondent had their first birth.

In [180]:
create_table_categorical("first_birth_age_group")

,segment,first_birth_age_group,percentage
1,R2,<18,6.954752
2,R3.1,<18,9.749163
3,R3.2,<18,8.974425
4,R4,<18,19.886221
5,U1,<18,7.101880
6,U2,<18,8.585293
7,U4,<18,27.092852
8,R2,>25,29.322633
9,R3.1,>25,24.063123
10,R3.2,>25,22.011672


In [181]:
create_figure_stacked_bars(create_table_categorical("first_birth_age_group"), "first_birth_age_group", "Age group at which the respondent had their first birth", "age group for first birth?")

#### 4.1.6. Ethnicity

In [182]:
create_table_categorical("ethnicity")

,segment,ethnicity,percentage
1,R2,Caste,16.932158
2,R3.1,Caste,5.224010
3,R3.2,Caste,0.882955
4,R4,Caste,0.325489
5,U1,Caste,13.158812
6,U2,Caste,4.644579
7,U4,Caste,2.089976
8,R2,No caste or tribe,0.000000
9,R3.1,No caste or tribe,0.000000
10,R3.2,No caste or tribe,0.000000


In [183]:
create_figure_stacked_bars(create_table_categorical("ethnicity"), "ethnicity", "Ethnicity", "Ethnicity?")

#### 4.1.7. Man's age at birth of first child

In [184]:
create_table("m.age.1stbirth", True)

,segment,m.age.1stbirth,percentage
1,R2,11,0.000000
2,R3.1,11,0.000000
3,R3.2,11,0.000000
4,R4,11,1.147581
5,U1,11,0.000000
...,...,...,...
213,R3.2,45,1.479744
214,R4,45,0.000000
215,U1,45,0.000000
216,U2,45,0.000000


In [185]:
create_figure_stacked_bars(create_table("m.age.1stbirth", True), "m.age.1stbirth", "Man's age at birth of first child", "Man's age at birth of first child?")

#### 4.1.8. Man's age at birth of first child

In [186]:
create_table_categorical("m.age.1stbirth.cat")

,segment,m.age.1stbirth.cat,percentage
1,R2,<18,0.000000
2,R3.1,<18,0.736856
3,R3.2,<18,0.000000
4,R4,<18,2.716710
5,U1,<18,0.000000
6,U2,<18,3.254915
7,U4,<18,0.000000
8,R2,>25,62.757443
9,R3.1,>25,33.904941
10,R3.2,>25,37.199048


In [187]:
create_figure_stacked_bars(create_table_categorical("m.age.1stbirth.cat"), "m.age.1stbirth.cat", "Man's age at birth of first child", "Man's age at birth of first child?")

#### 4.1.9. Man's age at birth of first child

In [188]:
create_table_categorical("m.age.1stbirth.cat1")

,segment,m.age.1stbirth.cat1,percentage
1,R2,<22,15.739727
2,R3.1,<22,30.257404
3,R3.2,<22,13.265757
4,R4,<22,33.288992
5,U1,<22,6.269128
6,U2,<22,22.771303
7,U4,<22,34.282257
8,R2,>28,37.568561
9,R3.1,>28,23.418147
10,R3.2,>28,21.519945


In [189]:
create_figure_stacked_bars(create_table_categorical("m.age.1stbirth.cat1"), "m.age.1stbirth.cat1", "Man's age at birth of first child", "Man's age at birth of first child?")

#### 4.1.10. Man's age at start of 1st marriage or union

In [190]:
create_table("m.age.1stcohab", True)

,segment,m.age.1stcohab,percentage
1,R2,No,0.000000
2,R3.1,No,0.000000
3,R3.2,No,0.000000
4,R4,No,0.000000
5,U1,No,0.000000
...,...,...,...
192,R3.2,42,2.060823
193,R4,42,0.000000
194,U1,42,0.000000
195,U2,42,0.000000


In [191]:
create_figure_stacked_bars(create_table("m.age.1stcohab", True), "m.age.1stcohab", "Man's age at start of 1st marriage or union", "Man's age at start of 1st marriage or union?")

#### 4.1.11. Man's age at start of 1st marriage or union

In [192]:
create_table_categorical("m.age.1stcohab.cat")

,segment,m.age.1stcohab.cat,percentage
1,R2,<18,4.078582
2,R3.1,<18,3.591820
3,R3.2,<18,6.917560
4,R4,<18,3.500538
5,U1,<18,0.000000
6,U2,<18,4.815901
7,U4,<18,0.000000
8,R2,>25,42.103467
9,R3.1,>25,32.540681
10,R3.2,>25,34.113949


In [193]:
create_figure_stacked_bars(create_table_categorical("m.age.1stcohab.cat"), "m.age.1stcohab.cat", "Man's age at start of 1st marriage or union", "Man's age at start of 1st marriage or union?")

#### 4.1.12. Man's age at start of 1st marriage or union

In [194]:
create_table_categorical("m.age.1stcohab.cat1")

,segment,m.age.1stcohab.cat1,percentage
1,R2,<20,7.926094
2,R3.1,<20,19.475701
3,R3.2,<20,17.310599
4,R4,<20,22.079514
5,U1,<20,3.134564
6,U2,<20,8.017886
7,U4,<20,34.282257
8,R2,>26,37.985009
9,R3.1,>26,31.026794
10,R3.2,>26,26.814955


In [195]:
create_figure_stacked_bars(create_table_categorical("m.age.1stcohab.cat1"), "m.age.1stcohab.cat1", "Man's age at start of 1st marriage or union", "Man's age at start of 1st marriage or union?")

#### 4.1.13. Man's (woman's partner or husband) age at 1st sex

In [196]:
create_table("m.age.1stsex", True)

,segment,m.age.1stsex,percentage
1,R2,Yes,0.000000
2,R3.1,Yes,0.000000
3,R3.2,Yes,0.000000
4,R4,Yes,0.431399
5,U1,Yes,0.000000
...,...,...,...
192,R3.2,98,1.219882
193,R4,98,0.684196
194,U1,98,0.828906
195,U2,98,1.244284


In [197]:
create_figure_stacked_bars(create_table("m.age.1stsex", True), "m.age.1stsex", "Man's (woman's partner or husband) age at 1st sex", "Man's (woman's partner or husband) age at 1st sex?")

#### 4.1.14. Man's age split at 1st sex

In [198]:
create_table_categorical("m.age.1stsex.cat")

,segment,m.age.1stsex.cat,percentage
1,R2,<18,17.490055
2,R3.1,<18,24.340496
3,R3.2,<18,19.718595
4,R4,<18,42.359623
5,U1,<18,3.134564
6,U2,<18,16.714320
7,U4,<18,84.155539
8,R2,>25,13.332294
9,R3.1,>25,13.854458
10,R3.2,>25,19.913007


In [199]:
create_figure_stacked_bars(create_table_categorical("m.age.1stsex.cat"), "m.age.1stsex.cat", "Man's age split at 1st sex", "Man's age split at 1st sex?")

#### 4.1.15. Wheter woman has expereienced stillbirth, abortion, and miscarriage

In [200]:
create_table("preg.loss", True)

,segment,preg.loss,percentage
1,R2,No,96.967469
2,R3.1,No,90.815401
3,R3.2,No,94.084378
4,R4,No,86.491993
5,U1,No,89.258395
6,U2,No,88.080865
7,U4,No,72.467406
8,R2,Yes,3.032531
9,R3.1,Yes,9.184599
10,R3.2,Yes,5.915622


In [201]:
create_figure_stacked_bars(create_table("preg.loss", True), "preg.loss", "Wheter woman has expereienced stillbirth, abortion, and miscarriage", "Wheter woman has expereienced stillbirth, abortion, and miscarriage?")

#### 4.1.16. Religion

In [202]:
create_table_categorical("religion")

,segment,religion,percentage
1,R2,Christian,81.904417
2,R3.1,Christian,87.194751
3,R3.2,Christian,87.283147
4,R4,Christian,90.086822
5,U1,Christian,96.463642
6,U2,Christian,86.729595
7,U4,Christian,84.928289
8,R2,Hindu,13.098610
9,R3.1,Hindu,5.489284
10,R3.2,Hindu,10.197828


In [203]:
create_figure_stacked_bars(create_table_categorical("religion"), "religion", "Religion", "Religion?")

#### 4.1.17. Woman's age at 1st birth (i.e., age at birth of first child)

In [204]:
create_table("w.age.1stbirth", True)

,segment,w.age.1stbirth,percentage
1,R2,11,0.230187
2,R3.1,11,0.000000
3,R3.2,11,0.000000
4,R4,11,0.000000
5,U1,11,0.000000
...,...,...,...
199,R3.2,39,0.000000
200,R4,39,0.000000
201,U1,39,0.000000
202,U2,39,0.280256


In [205]:
create_figure_stacked_bars(create_table("w.age.1stbirth", True), "w.age.1stbirth", "Woman's age at 1st birth (i.e., age at birth of first child)", "Woman's age at 1st birth (i.e., age at birth of first child)?")

#### 4.1.18. Woman's age split at 1st birth

In [206]:
create_table_categorical("w.age.1stbirth.cat")

,segment,w.age.1stbirth.cat,percentage
1,R2,<18,6.954752
2,R3.1,<18,9.749163
3,R3.2,<18,8.974425
4,R4,<18,19.886221
5,U1,<18,7.101880
6,U2,<18,8.585293
7,U4,<18,27.092852
8,R2,>25,19.968608
9,R3.1,>25,16.904470
10,R3.2,>25,16.885178


In [207]:
create_figure_stacked_bars(create_table_categorical("w.age.1stbirth.cat"), "w.age.1stbirth.cat", "Woman's age split at 1st birth", "Woman's age split at 1st birth?")

#### 4.1.19. Woman's age at start of 1st marriage or union

In [208]:
create_table("w.age.1stcohab", True)

,segment,w.age.1stcohab,percentage
1,R2,Yes,0.000000
2,R3.1,Yes,0.000000
3,R3.2,Yes,0.000000
4,R4,Yes,0.046475
5,U1,Yes,0.000000
...,...,...,...
206,R3.2,37,0.000000
207,R4,37,0.000000
208,U1,37,0.000000
209,U2,37,0.280256


In [209]:
create_figure_stacked_bars(create_table("w.age.1stcohab", True), "w.age.1stcohab", "Woman's age at start of 1st marriage or union", "Woman's age at start of 1st marriage or union?")

#### 4.1.20. Woman's age at start of 1st marriage or union

In [210]:
create_table_categorical("w.age.1stcohab.cat")

,segment,w.age.1stcohab.cat,percentage
1,R2,<18,15.859134
2,R3.1,<18,23.817831
3,R3.2,<18,23.335719
4,R4,<18,36.659953
5,U1,<18,11.688574
6,U2,<18,16.157967
7,U4,<18,56.640664
8,R2,>25,12.755524
9,R3.1,>25,10.606895
10,R3.2,>25,10.474787


In [211]:
create_figure_stacked_bars(create_table_categorical("w.age.1stcohab.cat"), "w.age.1stcohab.cat", "Woman's age at start of 1st marriage or union", "Woman's age at start of 1st marriage or union?")

#### 4.1.21. Woman's age at 1st sex

In [212]:
create_table("w.age.1stsex", True)

,segment,w.age.1stsex,percentage
1,R2,Yes,0.000000
2,R3.1,Yes,0.000000
3,R3.2,Yes,0.000000
4,R4,Yes,0.046475
5,U1,Yes,0.000000
...,...,...,...
220,R3.2,98,3.392224
221,R4,98,1.652604
222,U1,98,3.358422
223,U2,98,4.195023


In [213]:
create_figure_stacked_bars(create_table("w.age.1stsex", True), "w.age.1stsex", "Woman's age at 1st sex", "Woman's age at 1st sex?")

#### 4.1.22. Woman's age split at 1st sex

In [214]:
create_table_categorical("w.age.1stsex.cat")

,segment,w.age.1stsex.cat,percentage
1,R2,<18,15.445570
2,R3.1,<18,21.999428
3,R3.2,<18,18.623111
4,R4,<18,34.331608
5,U1,<18,10.988687
6,U2,<18,14.448827
7,U4,<18,46.883458
8,R2,>25,19.815112
9,R3.1,>25,15.558965
10,R3.2,>25,18.783333


In [215]:
create_figure_stacked_bars(create_table_categorical("w.age.1stsex.cat"), "w.age.1stsex.cat", "Woman's age split at 1st sex", "Woman's age split at 1st sex?")

### 4.2. Household relationships

#### 4.2.1. Number of adults in the household (4 or more).

In [216]:
create_table("cohab.adult.sum.4plus", True)

,segment,cohab.adult.sum.4plus,percentage
1,R2,No,85.008483
2,R3.1,No,70.844460
3,R3.2,No,85.571226
4,R4,No,79.455714
5,U1,No,83.159032
6,U2,No,45.140792
7,U4,No,67.940054
8,R2,Yes,14.991517
9,R3.1,Yes,29.155540
10,R3.2,Yes,14.428774


In [217]:
create_figure_stacked_bars(create_table("cohab.adult.sum.4plus", True), "cohab.adult.sum.4plus", "The household has 4 adults or more.", "4 adults or more?")

#### 4.2.2. Number of children currently alive.

In [218]:
create_table_categorical("num.child.alive")

,segment,num.child.alive,percentage
1,R2,0,0.000000
2,R3.1,0,0.123996
3,R3.2,0,1.862826
4,R4,0,0.000000
5,U1,0,0.000000
6,U2,0,0.000000
7,U4,0,0.000000
8,R2,1,33.249437
9,R3.1,1,32.887763
10,R3.2,1,39.626721


In [219]:
create_figure_stacked_bars(create_table_categorical("num.child.alive"), "num.child.alive", "Number of children currently alive", "Children currently alive?")

#### 4.2.3. The household head is male.

In [220]:
create_table("head.male", True)

,segment,head.male,percentage
1,R2,No,12.938962
2,R3.1,No,38.182494
3,R3.2,No,12.595185
4,R4,No,57.210592
5,U1,No,26.720892
6,U2,No,32.084084
7,U4,No,43.958004
8,R2,Yes,87.061038
9,R3.1,Yes,61.817506
10,R3.2,Yes,87.404815


In [221]:
create_figure_stacked_bars(create_table("head.male", True), "head.male", "The household head is male", "the household head is male?")

#### 4.2.4. Number of children alive.

In [222]:
create_table_categorical("num.children.cat")

,segment,num.children.cat,percentage
1,R2,0,0.000000
2,R3.1,0,0.123996
3,R3.2,0,1.862826
4,R4,0,0.000000
5,U1,0,0.000000
6,U2,0,0.000000
7,U4,0,0.000000
8,R2,1,33.249437
9,R3.1,1,32.887763
10,R3.2,1,39.626721


In [223]:
create_figure_stacked_bars(create_table_categorical("num.children.cat"), "num.children.cat", "Number of children alive", "Children alive?")

#### 4.2.5. Number of births the respondent has had.

In [224]:
create_table_categorical("num.births.cat")

,segment,num.births.cat,percentage
1,R2,1,33.041552
2,R3.1,1,32.533497
3,R3.2,1,39.312172
4,R4,1,10.044872
5,U1,1,14.227438
6,U2,1,28.939191
7,U4,1,10.812110
8,R2,2,41.899862
9,R3.1,2,32.321771
10,R3.2,2,36.671913


In [225]:
create_figure_stacked_bars(create_table_categorical("num.births.cat"), "num.births.cat", "Number of births the respondent has had", "births the respondent has had?")

#### 4.2.6. Pregnancies the respondent has had.


In [226]:
create_table_categorical("num.preg.cat")

,segment,num.preg.cat,percentage
1,R2,1,32.047671
2,R3.1,1,30.641822
3,R3.2,1,36.530097
4,R4,1,9.151560
5,U1,1,12.871091
6,U2,1,23.995322
7,U4,1,8.852232
8,R2,2,42.217934
9,R3.1,2,31.050858
10,R3.2,2,38.249910


In [227]:
create_figure_stacked_bars(create_table_categorical("num.preg.cat"), "num.preg.cat", "Pregnancies the respondent has had", "number of pregnancies the respondent has had?")

#### 4.2.7. Ratio of kids U5 to women 15-49 in the HH

In [228]:
create_table_categorical("hh.kidwom.rat.cat")

,segment,hh.kidwom.rat.cat,percentage
1,R2,Less than 1,80.148638
2,R3.1,Less than 1,73.348410
3,R3.2,Less than 1,76.175855
4,R4,Less than 1,45.683117
5,U1,Less than 1,74.383703
6,U2,Less than 1,78.593141
7,U4,Less than 1,57.148883
8,R2,More than 1,19.851362
9,R3.1,More than 1,26.651590
10,R3.2,More than 1,23.824145


In [229]:
create_figure_stacked_bars(create_table_categorical("hh.kidwom.rat.cat"), "hh.kidwom.rat.cat", "Ratio of kids U5 to women 15-49 in the HH", "Ratio of kids U5 to women 15-49 in the HH?")

#### 4.2.8. Made joined decision on family planning

In [230]:
create_table("jd.fp", True)

,segment,jd.fp,percentage
1,R2,No,22.635879
2,R3.1,No,27.370275
3,R3.2,No,17.492524
4,R4,No,11.234083
5,U1,No,24.468063
6,U2,No,5.399063
7,U4,No,15.556568
8,R2,Yes,77.364121
9,R3.1,Yes,72.629725
10,R3.2,Yes,82.507476


In [231]:
create_figure_stacked_bars(create_table("jd.fp", True), "jd.fp", "Made joined decision on family planning", "Made joined decision on family planning?")

#### 4.2.9. Made alone decision by herself on family planning

In [232]:
create_table("jd.fp", True)

,segment,jd.fp,percentage
1,R2,No,22.635879
2,R3.1,No,27.370275
3,R3.2,No,17.492524
4,R4,No,11.234083
5,U1,No,24.468063
6,U2,No,5.399063
7,U4,No,15.556568
8,R2,Yes,77.364121
9,R3.1,Yes,72.629725
10,R3.2,Yes,82.507476


In [233]:
create_figure_stacked_bars(create_table("jd.fp", True), "jd.fp", "Made alone decision by herself on family planning", "Made alone decision by herself on family planning?")

#### 4.2.10. Made joined decision on family planning

In [234]:
create_table_categorical("jd.fp.cat")

,segment,jd.fp.cat,percentage
1,R2,No,20.926034
2,R3.1,No,18.973881
3,R3.2,No,13.977076
4,R4,No,7.754195
5,U1,No,20.436135
6,U2,No,4.453320
7,U4,No,13.458066
8,R2,Pregnant,7.553693
9,R3.1,Pregnant,30.677054
10,R3.2,Pregnant,20.096861


In [235]:
create_figure_stacked_bars(create_table_categorical("jd.fp.cat"), "jd.fp.cat", "Made joined decision on family planning", "Made joined decision on family planning?")

#### 4.2.11. Made jalone decision on family planning

In [236]:
create_table_categorical("jd.fp.cat")

,segment,jd.fp.cat,percentage
1,R2,No,20.926034
2,R3.1,No,18.973881
3,R3.2,No,13.977076
4,R4,No,7.754195
5,U1,No,20.436135
6,U2,No,4.453320
7,U4,No,13.458066
8,R2,Pregnant,7.553693
9,R3.1,Pregnant,30.677054
10,R3.2,Pregnant,20.096861


In [237]:
create_figure_stacked_bars(create_table_categorical("jd.fp.cat"), "jd.fp.cat", "Made jalone decision on family planning", "Made jalone decision on family planning?")

#### 4.2.12. Made joined decision on partner's income

In [238]:
create_table("jd.hisincome", True)

,segment,jd.hisincome,percentage
1,R2,No,25.577843
2,R3.1,No,9.175050
3,R3.2,No,40.668094
4,R4,No,24.169789
5,U1,No,7.887544
6,U2,No,47.468802
7,U4,No,19.907838
8,R2,Yes,74.422157
9,R3.1,Yes,90.824950
10,R3.2,Yes,59.331906


In [239]:
create_figure_stacked_bars(create_table("jd.hisincome", True), "jd.hisincome", "Made joined decision on partner's income", "Made joined decision on partner's income?")

#### 4.2.13. Made joined decision on partner's income

In [240]:
create_table_categorical("jd.hisincome.cat")

,segment,jd.hisincome.cat,percentage
1,R2,No,25.577843
2,R3.1,No,9.175050
3,R3.2,No,40.668094
4,R4,No,24.169789
5,U1,No,7.887544
6,U2,No,47.468802
7,U4,No,19.907838
8,R2,Yes,74.422157
9,R3.1,Yes,90.824950
10,R3.2,Yes,59.331906


In [241]:
create_figure_stacked_bars(create_table_categorical("jd.hisincome.cat"), "jd.hisincome.cat", "Made joined decision on partner's income", "Made joined decision on partner's income?")

#### 4.2.14. Made joined decision on woman's health

In [242]:
create_table("jd.hlth", True)

,segment,jd.hlth,percentage
1,R2,No,20.192082
2,R3.1,No,7.533736
3,R3.2,No,38.513593
4,R4,No,12.922898
5,U1,No,20.897176
6,U2,No,19.961968
7,U4,No,5.665630
8,R2,Yes,79.807918
9,R3.1,Yes,92.466264
10,R3.2,Yes,61.486407


In [243]:
create_figure_stacked_bars(create_table("jd.hlth", True), "jd.hlth", "Made joined decision on woman's health", "Made joined decision on woman's health?")

#### 4.2.15. Made joined decision on woman's health

In [244]:
create_table_categorical("jd.hlth.cat")

,segment,jd.hlth.cat,percentage
1,R2,No,20.192082
2,R3.1,No,7.533736
3,R3.2,No,38.513593
4,R4,No,12.922898
5,U1,No,20.897176
6,U2,No,19.961968
7,U4,No,5.665630
8,R2,Yes,79.807918
9,R3.1,Yes,92.466264
10,R3.2,Yes,61.486407


In [245]:
create_figure_stacked_bars(create_table_categorical("jd.hlth.cat"), "jd.hlth.cat", "Made joined decision on woman's health", "Made joined decision on woman's health?")

#### 4.2.16. Woman participates in 3 joined decisions

In [246]:
create_table("jd.index.3", True)

,segment,jd.index.3,percentage
1,R2,No,53.034820
2,R3.1,No,100.000000
3,R3.2,No,100.000000
4,R4,No,96.087258
5,U1,No,97.623428
6,U2,No,33.447350
7,R2,Yes,46.965180
8,R3.1,Yes,0.000000
9,R3.2,Yes,0.000000
10,R4,Yes,3.912742


In [247]:
create_figure_stacked_bars(create_table("jd.index.3", True), "jd.index.3", "Woman participates in 3 joined decisions", "Woman participates in 3 joined decisions?")

#### 4.2.17. Woman participates in 3+ joined decisions

In [248]:
create_table("jd.index.3plus", True)

,segment,jd.index.3plus,percentage
1,R2,No,0.00000
2,R3.1,No,6.26291
3,R3.2,No,100.00000
4,R4,No,0.00000
5,U1,No,0.00000
6,U2,No,0.00000
7,R2,Yes,100.00000
8,R3.1,Yes,93.73709
9,R3.2,Yes,0.00000
10,R4,Yes,100.00000


In [249]:
create_figure_stacked_bars(create_table("jd.index.3plus", True), "jd.index.3plus", "Woman participates in 3+ joined decisions", "Woman participates in 3+ joined decisions?")

#### 4.2.18. Made joined decision on HH large purchase

In [250]:
create_table("jd.lrgpur", True)

,segment,jd.lrgpur,percentage
1,R2,No,42.637478
2,R3.1,No,8.479804
3,R3.2,No,33.843268
4,R4,No,18.618155
5,U1,No,31.181010
6,U2,No,25.027590
7,U4,No,41.332798
8,R2,Yes,57.362522
9,R3.1,Yes,91.520196
10,R3.2,Yes,66.156732


In [251]:
create_figure_stacked_bars(create_table("jd.lrgpur", True), "jd.lrgpur", "Made joined decision on HH large purchase", "Made joined decision on HH large purchase?")

#### 4.2.19. Made joined decision on HH large purchase

In [252]:
create_table_categorical("jd.lrgpur.cat")

,segment,jd.lrgpur.cat,percentage
1,R2,No,42.637478
2,R3.1,No,8.479804
3,R3.2,No,33.843268
4,R4,No,18.618155
5,U1,No,31.181010
6,U2,No,25.027590
7,U4,No,41.332798
8,R2,Yes,57.362522
9,R3.1,Yes,91.520196
10,R3.2,Yes,66.156732


In [253]:
create_figure_stacked_bars(create_table_categorical("jd.lrgpur.cat"), "jd.lrgpur.cat", "Made joined decision on HH large purchase", "Made joined decision on HH large purchase?")

#### 4.2.20. Made joined decision on woman's own income

In [254]:
create_table("jd.ownincome", True)

,segment,jd.ownincome,percentage
1,R2,No,19.208831
2,R3.1,No,2.985271
3,R3.2,No,34.451337
4,R4,No,19.728659
5,U1,No,15.417654
6,U2,No,23.285291
7,U4,No,0.000000
8,R2,Yes,80.791169
9,R3.1,Yes,97.014729
10,R3.2,Yes,65.548663


In [255]:
create_figure_stacked_bars(create_table("jd.ownincome", True), "jd.ownincome", "Made joined decision on woman's own income", "Made joined decision on woman's own income?")

#### 4.2.21. Made joined decision on woman's own income

In [256]:
create_table_categorical("jd.ownincome.cat")

,segment,jd.ownincome.cat,percentage
1,R2,No,7.178593
2,R3.1,No,1.271017
3,R3.2,No,14.208475
4,R4,No,6.660789
5,U1,No,8.256915
6,U2,No,11.771382
7,U4,No,0.000000
8,R2,Not paid in cash or not working,62.628682
9,R3.1,Not paid in cash or not working,57.423722
10,R3.2,Not paid in cash or not working,58.757842


In [257]:
create_figure_stacked_bars(create_table_categorical("jd.ownincome.cat"), "jd.ownincome.cat", "Made joined decision on woman's own income", "Made joined decision on woman's own income?")

#### 4.2.22. Made joined decision on visit to family or relatives

In [258]:
create_table("jd.visit", True)

,segment,jd.visit,percentage
1,R2,No,10.467467
2,R3.1,No,11.640262
3,R3.2,No,35.870611
4,R4,No,13.266566
5,U1,No,0.828906
6,U2,No,24.004243
7,U4,No,0.000000
8,R2,Yes,89.532533
9,R3.1,Yes,88.359738
10,R3.2,Yes,64.129389


In [259]:
create_figure_stacked_bars(create_table("jd.visit", True), "jd.visit", "Made joined decision on visit to family or relatives", "Made joined decision on visit to family or relatives?")

#### 4.2.23. Made joined decision on visit to family or relatives

In [260]:
create_table_categorical("jd.visit.cat")

,segment,jd.visit.cat,percentage
1,R2,No,10.467467
2,R3.1,No,11.640262
3,R3.2,No,35.870611
4,R4,No,13.266566
5,U1,No,0.828906
6,U2,No,24.004243
7,U4,No,0.000000
8,R2,Yes,89.532533
9,R3.1,Yes,88.359738
10,R3.2,Yes,64.129389


In [261]:
create_figure_stacked_bars(create_table_categorical("jd.visit.cat"), "jd.visit.cat", "Made joined decision on visit to family or relatives", "Made joined decision on visit to family or relatives?")

#### 4.2.24. Made either alone or joined decision on family planning

In [262]:
create_table("jdwd.fp", True)

,segment,jdwd.fp,percentage
1,R2,No,13.851155
2,R3.1,No,5.387519
3,R3.2,No,13.630847
4,R4,No,3.378378
5,U1,No,18.938171
6,U2,No,2.383074
7,U4,No,4.230430
8,R2,Yes,86.148845
9,R3.1,Yes,94.612481
10,R3.2,Yes,86.369153


In [263]:
create_figure_stacked_bars(create_table("jdwd.fp", True), "jdwd.fp", "Made either alone or joined decision on family planning", "Made either alone or joined decision on family planning?")

#### 4.2.25. Made either alone or joined decision on partner's income

In [264]:
create_table("jdwd.hisincome", True)

,segment,jdwd.hisincome,percentage
1,R2,No,24.705992
2,R3.1,No,9.175050
3,R3.2,No,35.082303
4,R4,No,14.306681
5,U1,No,7.887544
6,U2,No,43.232035
7,U4,No,0.000000
8,R2,Yes,75.294008
9,R3.1,Yes,90.824950
10,R3.2,Yes,64.917697


In [265]:
create_figure_stacked_bars(create_table("jdwd.hisincome", True), "jdwd.hisincome", "Made either alone or joined decision on partner's income", "Made either alone or joined decision on partner's income?")

#### 4.2.26. Made either alone or joined decision on wman's health

In [266]:
create_table("jdwd.hlth", True)

,segment,jdwd.hlth,percentage
1,R2,No,7.869876
2,R3.1,No,2.055125
3,R3.2,No,33.809049
4,R4,No,2.925572
5,U1,No,0.828906
6,U2,No,7.467412
7,U4,No,0.000000
8,R2,Yes,92.130124
9,R3.1,Yes,97.944875
10,R3.2,Yes,66.190951


In [267]:
create_figure_stacked_bars(create_table("jdwd.hlth", True), "jdwd.hlth", "Made either alone or joined decision on wman's health", "Made either alone or joined decision on wman's health?")

#### 4.2.27. Woman participantes in 3 decisions, either alone or joined

In [268]:
create_table("jdwd.index.3", True)

,segment,jdwd.index.3,percentage
1,R2,No,88.857212
2,R3.1,No,93.737090
3,R3.2,No,100.000000
4,R4,No,100.000000
5,U1,No,100.000000
6,U2,No,100.000000
7,R2,Yes,11.142788
8,R3.1,Yes,6.262910
9,R3.2,Yes,0.000000
10,R4,Yes,0.000000


In [269]:
create_figure_stacked_bars(create_table("jdwd.index.3", True), "jdwd.index.3", "Woman participantes in 3 decisions, either alone or joined", "Woman participantes in 3 decisions, either alone or joined?")

#### 4.2.28. Woman participantes in 3 decisions, either alone or joined

In [270]:
create_table("jdwd.index.3plus", True)

,segment,jdwd.index.3plus,percentage
1,R2,No,0.0
2,R3.1,No,0.0
3,R3.2,No,100.0
4,R4,No,0.0
5,U1,No,0.0
6,U2,No,0.0
7,R2,Yes,100.0
8,R3.1,Yes,100.0
9,R3.2,Yes,0.0
10,R4,Yes,100.0


In [271]:
create_figure_stacked_bars(create_table("jdwd.index.3plus", True), "jdwd.index.3plus", "Woman participantes in 3 decisions, either alone or joined", "Woman participantes in 3 decisions, either alone or joined?")

#### 4.2.29. Made either alone or joined decision on HH large purchase

In [272]:
create_table("jdwd.lrgpur", True)

,segment,jdwd.lrgpur,percentage
1,R2,No,17.027869
2,R3.1,No,2.702784
3,R3.2,No,30.274730
4,R4,No,7.913152
5,U1,No,0.828906
6,U2,No,14.797628
7,U4,No,0.000000
8,R2,Yes,82.972131
9,R3.1,Yes,97.297216
10,R3.2,Yes,69.725270


In [273]:
create_figure_stacked_bars(create_table("jdwd.lrgpur", True), "jdwd.lrgpur", "Made either alone or joined decision on HH large purchase", "Made either alone or joined decision on HH large purchase?")

#### 4.2.30. Made either alone or joined decision on woman's own income

In [274]:
create_table("jdwd.ownincome", True)

,segment,jdwd.ownincome,percentage
1,R2,No,11.522552
2,R3.1,No,2.985271
3,R3.2,No,29.055159
4,R4,No,8.313377
5,U1,No,0.000000
6,U2,No,12.177966
7,U4,No,0.000000
8,R2,Yes,88.477448
9,R3.1,Yes,97.014729
10,R3.2,Yes,70.944841


In [275]:
create_figure_stacked_bars(create_table("jdwd.ownincome", True), "jdwd.ownincome", "Made either alone or joined decision on woman's own income", "Made either alone or joined decision on woman's own income?")

#### 4.2.31. Made either alone or joined decision on visit to family or relatives

In [276]:
create_table("jdwd.visit", True)

,segment,jdwd.visit,percentage
1,R2,No,10.467467
2,R3.1,No,4.333868
3,R3.2,No,30.595594
4,R4,No,4.759034
5,U1,No,0.828906
6,U2,No,1.311102
7,U4,No,0.000000
8,R2,Yes,89.532533
9,R3.1,Yes,95.666132
10,R3.2,Yes,69.404406


In [277]:
create_figure_stacked_bars(create_table("jdwd.visit", True), "jdwd.visit", "Made either alone or joined decision on visit to family or relatives", "Made either alone or joined decision on visit to family or relatives?")

#### 4.2.32. Total number of living children U15

In [278]:
create_table("kids.under15", True)

,segment,kids.under15,percentage
1,R2,No,0.000000
2,R3.1,No,0.123996
3,R3.2,No,1.862826
4,R4,No,0.130527
5,U1,No,6.718612
...,...,...,...
59,R3.2,8,1.159149
60,R4,8,0.535258
61,U1,8,0.000000
62,U2,8,0.000000


In [279]:
create_figure_stacked_bars(create_table("kids.under15", True), "kids.under15", "Total number of living children U15", "Total number of living children U15?")

#### 4.2.33. Total number of living children U15

In [280]:
create_table_categorical("kids.under15.cat")

,segment,kids.under15.cat,percentage
1,R2,Four and more,7.043812
2,R3.1,Four and more,12.683346
3,R3.2,Four and more,6.912088
4,R4,Four and more,42.972368
5,U1,Four and more,7.267373
6,U2,Four and more,13.678486
7,U4,Four and more,33.962141
8,R2,One kid,36.088053
9,R3.1,One kid,33.491008
10,R3.2,One kid,41.211811


In [281]:
create_figure_stacked_bars(create_table_categorical("kids.under15.cat"), "kids.under15.cat", "Total number of living children U15", "Total number of living children U15?")

#### 4.2.34. Total number of living children U5

In [282]:
create_table("kids.under5", True)

,segment,kids.under5,percentage
1,R2,No,0.207885
2,R3.1,No,0.532164
3,R3.2,No,1.862826
4,R4,No,0.735378
5,U1,No,52.343679
6,U2,No,57.253964
7,U4,No,38.578827
8,R2,Yes,85.959171
9,R3.1,Yes,75.261678
10,R3.2,Yes,83.816445


In [283]:
create_figure_stacked_bars(create_table("kids.under5", True), "kids.under5", "Total number of living children U5", "Total number of living children U5?")

#### 4.2.35. Total number of living children 

In [284]:
create_table("num.children", True)

,segment,num.children,percentage
1,R2,No,0.000000
2,R3.1,No,0.123996
3,R3.2,No,1.862826
4,R4,No,0.000000
5,U1,No,0.000000
...,...,...,...
94,R3.2,13,0.000000
95,R4,13,0.039300
96,U1,13,0.000000
97,U2,13,0.000000


In [285]:
create_figure_stacked_bars(create_table("num.children", True), "num.children", "Total number of living children ", "Total number of living children ?")

#### 4.2.36. Total number of U5 children resident at home

In [286]:
create_table_categorical("num.u5.home.cat")

,segment,num.u5.home.cat,percentage
1,R2,0,0.207885
2,R3.1,0,1.206825
3,R3.2,0,2.044410
4,R4,0,0.983094
5,U1,0,45.700500
6,U2,0,50.272702
7,U4,0,19.211581
8,R2,1,78.215492
9,R3.1,1,64.722992
10,R3.2,1,72.544963


In [287]:
create_figure_stacked_bars(create_table_categorical("num.u5.home.cat"), "num.u5.home.cat", "Total number of U5 children resident at home", "Total number of U5 children resident at home?")

#### 4.2.37. Total number of U5 children resident at home

In [288]:
create_table("num.under5.2plus", True)

,segment,num.under5.2plus,percentage
1,R2,No,78.423378
2,R3.1,No,65.929817
3,R3.2,No,74.589373
4,R4,No,39.315208
5,U1,No,85.220015
6,U2,No,86.263788
7,U4,No,60.216056
8,R2,Yes,21.576622
9,R3.1,Yes,34.070183
10,R3.2,Yes,25.410627


In [289]:
create_figure_stacked_bars(create_table("num.under5.2plus", True), "num.under5.2plus", "Total number of U5 children resident at home", "Total number of U5 children resident at home?")

#### 4.2.38. Status of partner absence

In [290]:
create_table_categorical("partner.absent.cat")

,segment,partner.absent.cat,percentage
1,R2,No,98.625019
2,R3.1,No,95.399113
3,R3.2,No,100.000000
4,R4,No,97.042558
5,U1,No,97.390432
6,U2,No,94.657936
7,U4,No,95.713426
8,R2,Yes,1.374981
9,R3.1,Yes,4.600887
10,R3.2,Yes,0.000000


In [291]:
create_figure_stacked_bars(create_table_categorical("partner.absent.cat"), "partner.absent.cat", "Status of partner absence", "Status of partner absence?")

#### 4.2.39. Total number of children ever born

In [292]:
create_table("tot.livebirth", True)

,segment,tot.livebirth,percentage
1,R2,Yes,33.041552
2,R3.1,Yes,32.533497
3,R3.2,Yes,39.312172
4,R4,Yes,10.044872
5,U1,Yes,14.227438
...,...,...,...
87,R3.2,13,0.000000
88,R4,13,0.039300
89,U1,13,0.000000
90,U2,13,0.000000


In [293]:
create_figure_stacked_bars(create_table("tot.livebirth", True), "tot.livebirth", "Total number of children ever born", "Total number of children ever born?")

#### 4.2.40. Women who worked in the last 12 months and were paid in cash

In [294]:
create_table_categorical("w.work.cash")

,segment,w.work.cash,percentage
1,R2,No,94.360059
2,R3.1,No,93.834375
3,R3.2,No,92.808148
4,R4,No,95.441773
5,U1,No,85.512707
6,U2,No,89.194003
7,U4,No,97.241107
8,R2,Yes,5.639941
9,R3.1,Yes,6.165625
10,R3.2,Yes,7.191852


In [295]:
create_figure_stacked_bars(create_table_categorical("w.work.cash"), "w.work.cash", "Women who worked in the last 12 months and were paid in cash", "Women who worked in the last 12 months and were paid in cash?")

#### 4.2.41. Made decision alone by herself on partner's income

In [296]:
create_table("wd.hisincome", True)

,segment,wd.hisincome,percentage
1,R2,No,99.128149
2,R3.1,No,100.000000
3,R3.2,No,94.414209
4,R4,No,90.136892
5,U1,No,100.000000
6,U2,No,95.763233
7,U4,No,80.092162
8,R2,Yes,0.871851
9,R3.1,Yes,0.000000
10,R3.2,Yes,5.585791


In [297]:
create_figure_stacked_bars(create_table("wd.hisincome", True), "wd.hisincome", "Made decision alone by herself on partner's income", "Made decision alone by herself on partner's income?")

#### 4.2.42. Made alone decision by herself on partner's income

In [298]:
create_table_categorical("wd.hisincome.cat")

,segment,wd.hisincome.cat,percentage
1,R2,No,99.128149
2,R3.1,No,100.000000
3,R3.2,No,94.414209
4,R4,No,90.136892
5,U1,No,100.000000
6,U2,No,95.763233
7,U4,No,80.092162
8,R2,Yes,0.871851
9,R3.1,Yes,0.000000
10,R3.2,Yes,5.585791


In [299]:
create_figure_stacked_bars(create_table_categorical("wd.hisincome.cat"), "wd.hisincome.cat", "Made alone decision by herself on partner's income", "Made alone decision by herself on partner's income?")

#### 4.2.43. Made decision alone by herself on woman's health

In [300]:
create_table("wd.hlth", True)

,segment,wd.hlth,percentage
1,R2,No,87.677794
2,R3.1,No,94.521389
3,R3.2,No,95.295456
4,R4,No,90.002674
5,U1,No,79.931730
6,U2,No,87.505444
7,U4,No,94.334370
8,R2,Yes,12.322206
9,R3.1,Yes,5.478611
10,R3.2,Yes,4.704544


In [301]:
create_figure_stacked_bars(create_table("wd.hlth", True), "wd.hlth", "Made decision alone by herself on woman's health", "Made decision alone by herself on woman's health?")

#### 4.2.44. Made alone decision by herself on woman's health

In [302]:
create_table_categorical("wd.hlth.cat")

,segment,wd.hlth.cat,percentage
1,R2,No,87.677794
2,R3.1,No,94.521389
3,R3.2,No,95.295456
4,R4,No,90.002674
5,U1,No,79.931730
6,U2,No,87.505444
7,U4,No,94.334370
8,R2,Yes,12.322206
9,R3.1,Yes,5.478611
10,R3.2,Yes,4.704544


In [303]:
create_figure_stacked_bars(create_table_categorical("wd.hlth.cat"), "wd.hlth.cat", "Made alone decision by herself on woman's health", "Made alone decision by herself on woman's health?")

#### 4.2.45. Woman makes 3 decisions on her own

In [304]:
create_table("wd.index.3", True)

,segment,wd.index.3,percentage
1,R2,No,100.000000
2,R3.1,No,100.000000
3,R3.2,No,100.000000
4,R4,No,96.087258
5,U1,No,97.623428
6,U2,No,100.000000
7,R2,Yes,0.000000
8,R3.1,Yes,0.000000
9,R3.2,Yes,0.000000
10,R4,Yes,3.912742


In [305]:
create_figure_stacked_bars(create_table("wd.index.3", True), "wd.index.3", "Woman makes 3 decisions on her own", "Woman makes 3 decisions on her own?")

#### 4.2.46. Woman makes 3+ decisions on her own

In [306]:
create_table("wd.index.3plus", True)

,wd.index.3plus,segment,percentage


In [307]:
create_figure_stacked_bars(create_table("wd.index.3plus", True), "wd.index.3plus", "Woman makes 3+ decisions on her own", "Woman makes 3+ decisions on her own?")

#### 4.2.47. Made decision alone by herself on HH large purchase

In [308]:
create_table("wd.lrgpur", True)

,segment,wd.lrgpur,percentage
1,R2,No,74.390391
2,R3.1,No,94.222979
3,R3.2,No,96.431462
4,R4,No,89.294997
5,U1,No,69.647896
6,U2,No,89.770038
7,U4,No,58.667202
8,R2,Yes,25.609609
9,R3.1,Yes,5.777021
10,R3.2,Yes,3.568538


In [309]:
create_figure_stacked_bars(create_table("wd.lrgpur", True), "wd.lrgpur", "Made decision alone by herself on HH large purchase", "Made decision alone by herself on HH large purchase?")

#### 4.2.48. Made alone decision by herself on HH large purchase

In [310]:
create_table_categorical("wd.lrgpur.cat")

,segment,wd.lrgpur.cat,percentage
1,R2,No,74.390391
2,R3.1,No,94.222979
3,R3.2,No,96.431462
4,R4,No,89.294997
5,U1,No,69.647896
6,U2,No,89.770038
7,U4,No,58.667202
8,R2,Yes,25.609609
9,R3.1,Yes,5.777021
10,R3.2,Yes,3.568538


In [311]:
create_figure_stacked_bars(create_table_categorical("wd.lrgpur.cat"), "wd.lrgpur.cat", "Made alone decision by herself on HH large purchase", "Made alone decision by herself on HH large purchase?")

#### 4.2.49. Made decision alone on woman's own income

In [312]:
create_table("wd.ownincome", True)

,segment,wd.ownincome,percentage
1,R2,No,92.313721
2,R3.1,No,100.000000
3,R3.2,No,94.603821
4,R4,No,88.584718
5,U1,No,84.582346
6,U2,No,88.892675
7,U4,No,100.000000
8,R2,Yes,7.686279
9,R3.1,Yes,0.000000
10,R3.2,Yes,5.396179


In [313]:
# create_figure_stacked_bars(create_table("wd.ownincome ", True), "wd.ownincome ", "Made decision alone on woman's own income", "Made decision alone on woman's own income?")

#### 4.2.50. Made decision alone by herself on woman's own income

In [314]:
create_table_categorical("wd.ownincome.cat")

,segment,wd.ownincome.cat,percentage
1,R2,No,34.498855
2,R3.1,No,42.576278
3,R3.2,No,39.016657
4,R4,No,29.907968
5,U1,No,45.298023
6,U2,No,44.937795
7,U4,No,24.517157
8,R2,Not paid in cash or not working,62.628682
9,R3.1,Not paid in cash or not working,57.423722
10,R3.2,Not paid in cash or not working,58.757842


In [315]:
create_figure_stacked_bars(create_table_categorical("wd.ownincome.cat"), "wd.ownincome.cat", "Made decision alone by herself on woman's own income", "Made decision alone by herself on woman's own income?")

#### 4.2.51. Made decision alone by herself on visit to family or relatives

In [316]:
create_table("wd.visit", True)

,segment,wd.visit,percentage
1,R2,No,100.000000
2,R3.1,No,92.693606
3,R3.2,No,94.724984
4,R4,No,91.492468
5,U1,No,100.000000
6,U2,No,77.306859
7,U4,No,100.000000
8,R2,Yes,0.000000
9,R3.1,Yes,7.306394
10,R3.2,Yes,5.275016


In [317]:
create_figure_stacked_bars(create_table("wd.visit", True), "wd.visit", "Made decision alone by herself on visit to family or relatives", "Made decision alone by herself on visit to family or relatives?")

#### 4.2.52. Made alone decision by herself on visit to family or relatives

In [318]:
create_table_categorical("wd.visit.cat")

,segment,wd.visit.cat,percentage
1,R2,No,100.000000
2,R3.1,No,92.693606
3,R3.2,No,94.724984
4,R4,No,91.492468
5,U1,No,100.000000
6,U2,No,77.306859
7,U4,No,100.000000
8,R2,Yes,0.000000
9,R3.1,Yes,7.306394
10,R3.2,Yes,5.275016


In [319]:
create_figure_stacked_bars(create_table_categorical("wd.visit.cat"), "wd.visit.cat", "Made alone decision by herself on visit to family or relatives", "Made alone decision by herself on visit to family or relatives?")

### 4.3.  Household economics

#### 4.3.1. The household has a washing machine.

In [320]:
create_table("hh.wash.machine", True)

,segment,hh.wash.machine,percentage
1,R2,No,98.963224
2,R3.1,No,98.557446
3,R3.2,No,100.000000
4,R4,No,99.922388
5,U1,No,92.039251
6,U2,No,77.100819
7,U4,No,100.000000
8,R2,Yes,1.036776
9,R3.1,Yes,1.442554
10,R3.2,Yes,0.000000


In [321]:
create_figure_stacked_bars(create_table("hh.wash.machine", True), "hh.wash.machine", "The household has a washing machine", "has a washing machine?")

#### 4.3.2. House ownership type.

In [322]:
create_table_categorical("hh.house.owner.cat")

,segment,hh.house.owner.cat,percentage
1,R2,both,5.762886
2,R3.1,both,23.200481
3,R3.2,both,30.668736
4,R4,both,29.377048
5,U1,both,17.576287
6,U2,both,23.327091
7,U4,both,14.040710
8,R2,female member,67.989335
9,R3.1,female member,61.368593
10,R3.2,female member,45.591847


In [323]:
create_figure_stacked_bars(create_table_categorical("hh.house.owner.cat"), "hh.house.owner.cat", "Type of house ownership", "house ownership?")

#### 4.3.3. The household owns land.

In [324]:
create_table("hh.land", True)

,segment,hh.land,percentage
1,R2,No,60.257616
2,R3.1,No,81.128071
3,R3.2,No,89.219340
4,R4,No,81.505032
5,U1,No,87.527061
6,U2,No,86.072806
7,U4,No,78.437787
8,R2,Yes,39.742384
9,R3.1,Yes,18.871929
10,R3.2,Yes,10.780660


In [325]:
create_figure_stacked_bars(create_table("hh.land", True), "hh.land", "The household owns land", "the household owns land?")

#### 4.3.4. Air conditioner.


In [326]:
create_table("hh.ac", True)

,segment,hh.ac,percentage
1,R2,No,100.000000
2,R3.1,No,99.712859
3,R3.2,No,100.000000
4,R4,No,100.000000
5,U1,No,100.000000
6,U2,No,95.013778
7,U4,No,100.000000
8,R2,Yes,0.000000
9,R3.1,Yes,0.287141
10,R3.2,Yes,0.000000


In [327]:
create_figure_stacked_bars(create_table("hh.ac", True), "hh.ac", "The household has an air conditioner", "the household has an air conditioner?").show()

#### 4.3.5. TV.


In [328]:
create_table("hh.tv", True)

,segment,hh.tv,percentage
1,R2,No,19.617878
2,R3.1,No,17.565444
3,R3.2,No,56.395399
4,R4,No,83.684415
5,U1,No,8.168853
6,U2,No,8.128630
7,U4,No,60.114978
8,R2,Yes,80.382122
9,R3.1,Yes,82.434556
10,R3.2,Yes,43.604601


In [329]:
create_figure_stacked_bars(create_table("hh.tv", True), "hh.tv", "The household has a TV", "the household has a TV?").show()

#### 4.3.6. Whether partner has professional job

In [330]:
create_table_categorical("m.work.prof")

,segment,m.work.prof,percentage
1,R2,Non professional,14.852529
2,R3.1,Non professional,13.744552
3,R3.2,Non professional,17.438109
4,R4,Non professional,13.320045
5,U1,Non professional,14.797582
6,U2,Non professional,15.974744
7,U4,Non professional,8.375151
8,R2,Not working or no partner,84.908370
9,R3.1,Not working or no partner,85.518638
10,R3.2,Not working or no partner,82.561891


In [331]:
create_figure_stacked_bars(create_table_categorical("m.work.prof"), "m.work.prof", "Whether partner has professional job", "Whether partner has professional job?")

#### 4.3.7. Whether partner has professional job

In [332]:
create_table("m.work.prof.yn", True)

,segment,m.work.prof.yn,percentage
1,R2,No,99.760899
2,R3.1,No,99.263190
3,R3.2,No,100.000000
4,R4,No,99.818986
5,U1,No,87.746308
6,U2,No,94.599108
7,U4,No,97.122241
8,R2,Yes,0.239101
9,R3.1,Yes,0.736810
10,R3.2,Yes,0.000000


In [333]:
create_figure_stacked_bars(create_table("m.work.prof.yn", True), "m.work.prof.yn", "Whether partner has professional job", "Whether partner has professional job?")

#### 4.3.8. Partner's work type

In [334]:
create_table_categorical("m.work.type")

,segment,m.work.type,percentage
1,R2,Agricultural,10.302019
2,R3.1,Agricultural,4.109756
3,R3.2,Agricultural,11.640682
4,R4,Agricultural,4.781025
5,U1,Agricultural,0.324149
6,U2,Agricultural,0.280256
7,U4,Agricultural,3.205326
8,R2,Industrial,1.752484
9,R3.1,Industrial,4.227434
10,R3.2,Industrial,0.981608


In [335]:
create_figure_stacked_bars(create_table_categorical("m.work.type"), "m.work.type", "Partner's work type", "Partner's work type?")

#### 4.3.9. Partner's work status

In [336]:
create_table_categorical("m.working")

,segment,m.working,percentage
1,R2,Currently working,10.866629
2,R3.1,Currently working,10.859532
3,R3.2,Currently working,11.299870
4,R4,Currently working,12.944363
5,U1,Currently working,26.502895
6,U2,Currently working,20.906850
7,U4,Currently working,9.012699
8,R2,Not working or no partner,85.892816
9,R3.1,Not working or no partner,88.517754
10,R3.2,Not working or no partner,88.522559


In [337]:
create_figure_stacked_bars(create_table_categorical("m.working"), "m.working", "Partner's work status", "Partner's work status?")

#### 4.3.10. Women having a bank or savings account that they themselves use

In [338]:
create_table_categorical("w.bank")

,segment,w.bank,percentage
1,R2,no,21.058927
2,R3.1,no,12.572882
3,R3.2,no,38.004575
4,R4,no,20.942966
5,U1,no,3.963470
6,U2,no,10.268478
7,U4,no,27.090590
8,R2,yes,78.941073
9,R3.1,yes,87.427118
10,R3.2,yes,61.995425


In [339]:
create_figure_stacked_bars(create_table_categorical("w.bank"), "w.bank", "Women having a bank or savings account that they themselves use", "Women having a bank or savings account that they themselves use?")

#### 4.3.11. Whether woman currently has a job from which she is absent

In [340]:
create_table("w.job.yn", True)

,segment,w.job.yn,percentage
1,R2,No,74.841115
2,R3.1,No,61.409215
3,R3.2,No,55.303983
4,R4,No,68.584884
5,U1,No,47.643339
6,U2,No,44.806323
7,U4,No,75.482843
8,R2,Yes,25.158885
9,R3.1,Yes,38.590785
10,R3.2,Yes,44.696017


In [341]:
create_figure_stacked_bars(create_table("w.job.yn", True), "w.job.yn", "Whether woman currently has a job from which she is absent", "Whether woman currently has a job from which she is absent?")

#### 4.3.12. Women having a mobile phone

In [342]:
create_table_categorical("w.mobile")

,segment,w.mobile,percentage
1,R2,no,11.604036
2,R3.1,no,16.825202
3,R3.2,no,53.158441
4,R4,no,44.417862
5,U1,no,20.897176
6,U2,no,2.756608
7,U4,no,37.365538
8,R2,yes,88.395964
9,R3.1,yes,83.174798
10,R3.2,yes,46.841559


In [343]:
create_figure_stacked_bars(create_table_categorical("w.mobile"), "w.mobile", "Women having a mobile phone", "Women having a mobile phone?")

#### 4.3.13. Woman uses her mobile telephone for financial transactions

In [344]:
create_table_categorical("w.mobile.trans")

,segment,w.mobile.trans,percentage
1,R2,no,96.727973
2,R3.1,no,90.183774
3,R3.2,no,84.921451
4,R4,no,94.000864
5,U1,no,79.590874
6,U2,no,81.259419
7,U4,no,100.000000
8,R2,yes,3.272027
9,R3.1,yes,9.816226
10,R3.2,yes,15.078549


In [345]:
create_figure_stacked_bars(create_table_categorical("w.mobile.trans"), "w.mobile.trans", "Woman uses her mobile telephone for financial transactions", "Woman uses her mobile telephone for financial transactions?")

#### 4.3.14. Women owning a house, either alone or jointly with others

In [346]:
create_table("w.own.house", True)

,segment,w.own.house,percentage
1,R2,No,23.509686
2,R3.1,No,31.243683
3,R3.2,No,19.833465
4,R4,No,14.090670
5,U1,No,20.897176
6,U2,No,26.172783
7,U4,No,5.665630
8,R2,Yes,76.490314
9,R3.1,Yes,68.756317
10,R3.2,Yes,80.166535


In [347]:
create_figure_stacked_bars(create_table("w.own.house", True), "w.own.house", "Women owning a house, either alone or jointly with others", "Women owning a house, either alone or jointly with others?")

#### 4.3.15. Women owning house or land, either alone or jointly with others

In [348]:
create_table("w.own.houseland", True)

,segment,w.own.houseland,percentage
1,R2,No,22.111509
2,R3.1,No,25.857842
3,R3.2,No,18.162279
4,R4,No,14.090670
5,U1,No,20.897176
6,U2,No,26.172783
7,U4,No,5.665630
8,R2,Yes,77.888491
9,R3.1,Yes,74.142158
10,R3.2,Yes,81.837721


In [349]:
create_figure_stacked_bars(create_table("w.own.houseland", True), "w.own.houseland", "Women owning house or land, either alone or jointly with others", "Women owning house or land, either alone or jointly with others?")

#### 4.3.16. Women owning land, either alone or jointly with others

In [350]:
create_table("w.own.land", True)

,segment,w.own.land,percentage
1,R2,No,51.451898
2,R3.1,No,38.592084
3,R3.2,No,31.999885
4,R4,No,45.281569
5,U1,No,93.730872
6,U2,No,81.541280
7,U4,No,85.757792
8,R2,Yes,48.548102
9,R3.1,Yes,61.407916
10,R3.2,Yes,68.000115


In [351]:
create_figure_stacked_bars(create_table("w.own.land", True), "w.own.land", "Women owning land, either alone or jointly with others", "Women owning land, either alone or jointly with others?")

#### 4.3.17. Woman's work frequency

In [352]:
create_table_categorical("w.work.freq")

,segment,w.work.freq,percentage
1,R2,All year,10.464034
2,R3.1,All year,27.240691
3,R3.2,All year,19.517009
4,R4,All year,21.124820
5,U1,All year,52.356661
6,U2,All year,35.532022
7,U4,All year,0.000000
8,R2,No work in the past year,56.710793
9,R3.1,No work in the past year,54.457714
10,R3.2,No work in the past year,48.995488


In [353]:
create_figure_stacked_bars(create_table_categorical("w.work.freq"), "w.work.freq", "Woman's work frequency", "Woman's work frequency?")

#### 4.3.18. Whether woman has a professional job

In [354]:
create_table_categorical("w.work.prof")

,segment,w.work.prof,percentage
1,R2,Non professional,43.289207
2,R3.1,Non professional,44.155694
3,R3.2,Non professional,49.090245
4,R4,Non professional,41.660010
5,U1,Non professional,24.401110
6,U2,Non professional,47.429980
7,U4,Non professional,30.182786
8,R2,Not working,56.710793
9,R3.1,Not working,54.457714
10,R3.2,Not working,48.995488


In [355]:
create_figure_stacked_bars(create_table_categorical("w.work.prof"), "w.work.prof", "Whether woman has a professional job", "Whether woman has a professional job?")

#### 4.3.19. Whether woman has a professional job

In [356]:
create_table("w.work.prof.yn", True)

,segment,w.work.prof.yn,percentage
1,R2,No,100.000000
2,R3.1,No,98.613407
3,R3.2,No,98.085732
4,R4,No,98.446257
5,U1,No,70.846173
6,U2,No,90.857999
7,U4,No,100.000000
8,R2,Yes,0.000000
9,R3.1,Yes,1.386593
10,R3.2,Yes,1.914268


In [357]:
create_figure_stacked_bars(create_table("w.work.prof.yn", True), "w.work.prof.yn", "Whether woman has a professional job", "Whether woman has a professional job?")

#### 4.3.20. Woman's occupation type

In [358]:
create_table_categorical("w.work.type")

,segment,w.work.type,percentage
1,R2,Agricultural,29.416769
2,R3.1,Agricultural,12.179976
3,R3.2,Agricultural,32.858019
4,R4,Agricultural,30.907620
5,U1,Agricultural,1.198277
6,U2,Agricultural,1.311102
7,U4,Agricultural,30.182786
8,R2,Industrial,8.631666
9,R3.1,Industrial,7.351011
10,R3.2,Industrial,0.000000


In [359]:
create_figure_stacked_bars(create_table_categorical("w.work.type"), "w.work.type", "Woman's occupation type", "Woman's occupation type?")

#### 4.3.21. Woman's work status

In [360]:
create_table_categorical("w.working")

,segment,w.working,percentage
1,R2,Currently working,25.158885
2,R3.1,Currently working,36.669697
3,R3.2,Currently working,43.558479
4,R4,Currently working,29.774919
5,U1,Currently working,52.356661
6,U2,Currently working,55.193677
7,U4,Currently working,24.517157
8,R2,No work in the past year,56.710793
9,R3.1,No work in the past year,54.457714
10,R3.2,No work in the past year,48.995488


In [361]:
create_figure_stacked_bars(create_table_categorical("w.working"), "w.working", "Woman's work status", "Woman's work status?")

### 4.4. Mental and healthcare models

#### 4.4.1. Oral Rehydration Solution (ORS).

In [362]:
create_table("heard.ors", True)

,segment,heard.ors,percentage
1,R2,No,5.119062
2,R3.1,No,4.128952
3,R3.2,No,16.754455
4,R4,No,1.731886
5,U1,No,0.356061
6,U2,No,2.939059
7,U4,No,1.559196
8,R2,Yes,94.880938
9,R3.1,Yes,95.871048
10,R3.2,Yes,83.245545


In [363]:
create_figure_stacked_bars(create_table("heard.ors", True), "heard.ors", "The woman has heard of Oral Rehydration Solution (ORS)", "the woman has heard of Oral Rehydration Solution (ORS)?").show()

#### 4.4.2. Ability to conceive

In [364]:
create_table("infecund.meno", True)

,segment,infecund.meno,percentage
1,R2,No,98.237549
2,R3.1,No,96.789309
3,R3.2,No,95.213805
4,R4,No,96.093385
5,U1,No,89.391670
6,U2,No,56.507596
7,U4,No,79.130495
8,R2,Yes,1.762451
9,R3.1,Yes,3.210691
10,R3.2,Yes,4.786195


In [365]:
create_figure_stacked_bars(create_table("infecund.meno", True), "infecund.meno", "Ability to conceive", "Ability to conceive?")

### 4.5. Life in the community

#### 4.5.1. The woman has been told about family planning by a CHW.

In [366]:
create_table("told.fp.chw", True)

,segment,told.fp.chw,percentage
1,R2,No,37.398283
2,R3.1,No,43.399931
3,R3.2,No,62.980035
4,R4,No,37.204927
5,U1,No,23.865841
6,U2,No,60.818353
7,U4,No,46.098320
8,R2,Yes,62.601717
9,R3.1,Yes,56.600069
10,R3.2,Yes,37.019965


In [367]:
create_figure_stacked_bars(create_table("told.fp.chw", True), "told.fp.chw", "The woman has been told about family planning by a CHW", "info about family planning by a CHW?")

#### 2.5.2. Family planning information from friends.

In [368]:
create_table("source.fp.friend", True)

,segment,source.fp.friend,percentage
1,R2,No,97.896527
2,R3.1,No,99.035253
3,R3.2,No,99.215744
4,R4,No,98.774487
5,U1,No,93.707722
6,U2,No,100.000000
7,U4,No,100.000000
8,R2,Yes,2.103473
9,R3.1,Yes,0.964747
10,R3.2,Yes,0.784256


In [369]:
create_figure_stacked_bars(create_table("source.fp.friend", True), "source.fp.friend", "The woman gets family planning information from friends", "the woman gets family planning information from friends?").show()

### 4.6. Larger environment

#### 4.6.1. The household has walls made of natural or rudimentary materials.

In [370]:
create_table("wall.slum", True)

,segment,wall.slum,percentage
1,R2,No,70.998654
2,R3.1,No,92.884561
3,R3.2,No,73.297065
4,R4,No,74.588392
5,U1,No,93.744984
6,U2,No,94.403769
7,U4,No,57.368002
8,R2,Yes,29.001346
9,R3.1,Yes,7.115439
10,R3.2,Yes,26.702935


In [371]:
create_figure_stacked_bars(create_table("wall.slum", True), "wall.slum", "The household has walls made of natural or rudimentary materials", "walls made of natural or rudimentary materials?")

#### 4.6.2. HH migration status

In [372]:
create_table_categorical("migrant")

,segment,migrant,percentage
1,R2,Always live here,77.903976
2,R3.1,Always live here,63.759592
3,R3.2,Always live here,74.159354
4,R4,Always live here,54.703907
5,U1,Always live here,61.483369
6,U2,Always live here,53.632510
7,U4,Always live here,48.246287
8,R2,Less than 1 year,0.000000
9,R3.1,Less than 1 year,1.276310
10,R3.2,Less than 1 year,2.323922


In [373]:
create_figure_stacked_bars(create_table_categorical("migrant"), "migrant", "HH migration status", "HH migration status?")

#### 4.6.3. HH migration status

In [374]:
create_table("migrant.yn", True)

,segment,migrant.yn,percentage
1,R2,No,77.903976
2,R3.1,No,63.759592
3,R3.2,No,74.159354
4,R4,No,54.703907
5,U1,No,61.483369
6,U2,No,53.632510
7,U4,No,48.246287
8,R2,Yes,22.096024
9,R3.1,Yes,36.240408
10,R3.2,Yes,25.840646


In [375]:
create_figure_stacked_bars(create_table("migrant.yn", True), "migrant.yn", "HH migration status", "HH migration status?")